# **ICU Mortality Prediction Model — Phase 5 (full validation and transfer bias)**
This notebook brings together, in two parts, the complete validation of Model A:
- **Part I** evaluates the model under the original outcome definition (`mortalidade_hospitalar`): fairness, classification metrics, post-hoc recalibration, local transparency (SHAP), the TRIPOD-AI/PROBAST checklist, and robustness with repeated admissions.
- **Part II** investigates transfer bias: it rebuilds the index cohort from the raw Excel with the adjusted outcome definition (transferred patients excluded), refits Phase 4's winning engines, and compares discrimination between the two definitions, with the DeLong test.

## **Part I — Full evaluation of Model A (original definition)**
Phase 4 selected, per variable set, the winning gradient boosting engine and measured discrimination, fine calibration, and clinical utility. Here, we evaluate:
- 1. **Fairness** — performance by subgroup (age, sex, admission type).
- 2. **Classification metrics** at three operational thresholds (Youden, ≈0.10, ≈0.20), with confusion matrix, sensitivity/specificity/PPV/NPV, accuracy/F1 (with caveat), and MCC.
- 3. **Post-hoc recalibration** — decided by the 95% CI (bootstrap) of the intercept/slope, fitted by Platt scaling on *out-of-fold* training predictions, never on the test set.
- 4. **Transparency (local SHAP)** — case-level explanations for illustrative patients of each set.
- 5. **TRIPOD-AI / PROBAST reporting** — checklist scaffolding, to be consolidated in the final document (Phase 6).
- 6. **Robustness** — re-inclusion of the ~1,270 non-index admissions, with cross-validation grouped by patient (GroupKFold), comparing findings to the index-admission cohort.

## 0 · Configuration and loading of Phase 4 artifacts

> **Reproducibility note.** The original study ran in Google Colab with the source spreadsheet on Google Drive. In this repository, place the input files in `data/` and adjust the paths accordingly.

In [ ]:
!pip -q install shap --upgrade

In [ ]:
import os, json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedGroupKFold, GroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss, roc_curve,
                              confusion_matrix, accuracy_score, f1_score, matthews_corrcoef)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import joblib

import xgboost as xgb
import lightgbm as lgb

# === Configuration ===
FASE2_DIR = 'outputs/phase2'
FASE4_DIR = 'outputs/phase4'
OUTDIR    = 'outputs/phase5'
os.makedirs(OUTDIR, exist_ok=True)

CSV_PATH      = f'{FASE2_DIR}/modelo_A_fase2.csv'
CONJ_PATH     = f'{FASE2_DIR}/modelo_A_sets_variaveis.json'
MELHORES_PATH = f'{FASE4_DIR}/modelo_A_fase4_melhores_motores.json'

ALVO = 'mortalidade_hospitalar'
CORTE_TEMPORAL = 2023
SEED = 42
N_TOP_DIAG = 20
N_BOOT = 1000

CATEGORICAS = ['Gender','AdmissionSourceName','AdmissionTypeName','AdmissionReasonName',
               'AdmissionMainDiagnosisName_grp','Chronic Health Status Name','Status COVID 19',
               'OrigemPagamento']

print('Configuration loaded.')

In [ ]:
# --- Reconstruction identical to Phase 4: base, sets, temporal validation, grouped diagnosis ---
df = pd.read_csv(CSV_PATH)
with open(CONJ_PATH, encoding='utf-8') as f:
    sets = json.load(f)

CONJUNTO_1 = sets['set_1_escores']
CONJUNTO_2 = sets['set_2_bruto']
BENCHMARK  = sets['benchmark']

if 'ano' not in df.columns:
    df['ano'] = pd.to_datetime(df['Month Year']).dt.year

GROUP_COL = 'patient_id' if 'patient_id' in df.columns else None
if GROUP_COL is None:
    df['patient_id'] = df.index
    GROUP_COL = 'patient_id'

treino_t = df[df['ano'] <  CORTE_TEMPORAL].copy()
teste_t  = df[df['ano'] >= CORTE_TEMPORAL].copy()

def agrupar_diagnostico_principal(treino, teste, col='AdmissionMainDiagnosisName'):
    col_grp = f'{col}_grp'
    top_categorias = treino[col].value_counts().nlargest(N_TOP_DIAG).index
    treino = treino.copy(); teste = teste.copy()
    treino[col_grp] = treino[col].where(treino[col].isin(top_categorias), 'outros')
    teste[col_grp]  = teste[col].where(teste[col].isin(top_categorias), 'outros')
    return treino, teste

treino_t, teste_t = agrupar_diagnostico_principal(treino_t, teste_t)

def usar_diagnostico_agrupado(set):
    return ['AdmissionMainDiagnosisName_grp' if c == 'AdmissionMainDiagnosisName' else c for c in set]

CONJUNTO_1 = usar_diagnostico_agrupado(CONJUNTO_1)
CONJUNTO_2 = usar_diagnostico_agrupado(CONJUNTO_2)
CONJUNTOS_VAR = {'Conjunto 1 (escores)': CONJUNTO_1, 'Conjunto 2 (brutos)': CONJUNTO_2}

print(f'Training: {len(treino_t)}  |  Test: {len(teste_t)}  (temporal validation, cutoff {CORTE_TEMPORAL})')

In [ ]:
def preparar_boosting(set, treino, teste):
    cats = [c for c in set if c in CATEGORICAS]
    nums = [c for c in set if c not in CATEGORICAS and c != 'Month Year']
    cols = cats + nums

    Xtr = treino[cols].copy()
    Xte = teste[cols].copy()

    for c in cats:
        Xtr[c] = Xtr[c].astype('object').fillna('Not reported')
        Xte[c] = Xte[c].astype('object').fillna('Not reported')
        categorias = pd.Index(Xtr[c].unique()).union(['Not reported'])
        Xte[c] = Xte[c].where(Xte[c].isin(categorias), 'Not reported')
        Xtr[c] = pd.Categorical(Xtr[c], categories=categorias)
        Xte[c] = pd.Categorical(Xte[c], categories=categorias)

    return Xtr, Xte, cats, nums

print('Preparation function (identical to Phase 4) defined.')

In [ ]:
# --- Load each set's winning engine (Phase 4, temporal validation) and generate predictions ---
with open(MELHORES_PATH, encoding='utf-8') as f:
    melhor_por_set = json.load(f)

def caminho_modelo_fase4(nome_set, nome_motor):
    nome_arquivo = f"modelo_{nome_set.split()[1]}_{nome_motor}.joblib".replace('(', '').replace(')', '')
    return f'{FASE4_DIR}/{nome_arquivo}'

modelos, dados_prep, predicoes = {}, {}, {}
for nome_set, nome_motor in melhor_por_set.items():
    Xtr, Xte, cats, nums = preparar_boosting(CONJUNTOS_VAR[nome_set], treino_t, teste_t)
    modelo = joblib.load(caminho_modelo_fase4(nome_set, nome_motor))
    p_teste = modelo.predict_proba(Xte)[:, 1]

    modelos[nome_set] = modelo
    dados_prep[nome_set] = {'Xtr': Xtr, 'Xte': Xte, 'cats': cats, 'nums': nums}
    predicoes[nome_set] = p_teste
    print(f'{nome_set}: engine = {nome_motor}  |  test AUROC = {roc_auc_score(teste_t[ALVO], p_teste):.4f}')

y_teste = teste_t[ALVO].values

## 1 · Fairness — performance by subgroup

Discrimination (AUROC) and calibration (Brier) of each set's winning engine, in the temporal validation,
stratified by age, sex, and admission type. The 95% confidence interval of the AUROC is obtained by
bootstrap (resampling the subgroup with replacement, 1,000 repetitions), the same technique used in
Section 3 for the calibration intercept/slope.

In [ ]:
def bootstrap_ic_auroc(y, p, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y)
    if len(np.unique(y)) < 2 or n < 10:
        return np.nan, np.nan
    aurocs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y[idx])) < 2:
            continue
        aurocs.append(roc_auc_score(y[idx], p[idx]))
    return np.percentile(aurocs, 2.5), np.percentile(aurocs, 97.5)


def faixa_etaria(idade):
    if idade < 65:
        return '< 65 anos'
    if idade < 80:
        return '65–79 anos'
    return '≥ 80 anos'


teste_t['faixa_etaria'] = teste_t['Age'].apply(faixa_etaria)

SUBGRUPOS = {
    'Age group': 'faixa_etaria',
    'Sexo': 'Gender',
    'Admission type': 'AdmissionTypeName',
}

linhas_equidade = []
for nome_set, p_teste in predicoes.items():
    for nome_dim, col in SUBGRUPOS.items():
        for categoria, sub in teste_t.groupby(col):
            idx = sub.index
            pos = teste_t.index.get_indexer(idx)
            y_sub, p_sub = y_teste[pos], p_teste[pos]
            if len(y_sub) < 10 or len(np.unique(y_sub)) < 2:
                auroc, brier, ic_lo, ic_hi = np.nan, np.nan, np.nan, np.nan
            else:
                auroc = roc_auc_score(y_sub, p_sub)
                brier = brier_score_loss(y_sub, p_sub)
                ic_lo, ic_hi = bootstrap_ic_auroc(y_sub, p_sub)
            linhas_equidade.append({
                'set': nome_set, 'dimensao': nome_dim, 'categoria': categoria,
                'n': len(y_sub), 'taxa_obito': y_sub.mean() if len(y_sub) else np.nan,
                'AUROC': auroc, 'AUROC_ic_low': ic_lo, 'AUROC_ic_high': ic_hi, 'Brier': brier,
            })

tab_equidade = pd.DataFrame(linhas_equidade)
equidade_path = f'{OUTDIR}/modelo_A_fase5_equidade.csv'
tab_equidade.to_csv(equidade_path, index=False, encoding='utf-8-sig')
print(f'Tabela de equidade salva: {equidade_path}')
print(tab_equidade.to_string(index=False))

In [ ]:
# --- Illustrative plot: AUROC by subgroup, with 95% CI, alongside the overall AUROC (temporal validation) ---
DIMENSOES_ORDEM = list(SUBGRUPOS.keys())

fig, axes = plt.subplots(1, len(predicoes), figsize=(7 * len(predicoes), 6), sharex=True)
if len(predicoes) == 1:
    axes = [axes]

linhas_auroc_geral = []
for eixo, (nome_set, p_teste) in zip(axes, predicoes.items()):
    sub = tab_equidade[tab_equidade['set'] == nome_set].copy()
    sub['dimensao'] = pd.Categorical(sub['dimensao'], categories=DIMENSOES_ORDEM, ordered=True)
    sub = sub.sort_values(['dimensao', 'categoria']).dropna(subset=['AUROC']).reset_index(drop=True)

    rotulos = [f"{row['dimensao']} — {row['categoria']}  (n={row['n']})" for _, row in sub.iterrows()]
    y_pos = np.arange(len(sub))
    erro = np.vstack([sub['AUROC'] - sub['AUROC_ic_low'], sub['AUROC_ic_high'] - sub['AUROC']])

    auroc_geral = roc_auc_score(y_teste, p_teste)

    eixo.errorbar(sub['AUROC'], y_pos, xerr=erro, fmt='o', color='#1F3864',
                  ecolor='#1F3864', elinewidth=1.5, capsize=3, markersize=6)
    linha_geral = eixo.axvline(auroc_geral, linestyle='--', color='gray', linewidth=1.2)
    linhas_auroc_geral.append(f'{nome_set}: overall AUROC (test) = {auroc_geral:.3f}')

    # horizontal reference lines between dimensions, to visually separate the blocks
    limites_dimensao = sub['dimensao'].ne(sub['dimensao'].shift()).to_numpy().nonzero()[0]
    for lim in limites_dimensao[1:]:
        eixo.axhline(lim - 0.5, color='#DDDDDD', linewidth=0.8)

    eixo.set_yticks(y_pos)
    eixo.set_yticklabels(rotulos, fontsize=9)
    eixo.invert_yaxis()
    eixo.set_xlabel('AUROC (IC 95%, bootstrap)')
    eixo.set_title(nome_set)
    eixo.set_xlim(0.5, 1.0)

# Single legend, below the whole figure — avoids overlapping any axis's points
legenda_texto = '   |   '.join(linhas_auroc_geral)
fig.legend([linha_geral], [legenda_texto], loc='lower center', bbox_to_anchor=(0.5, -0.04),
           fontsize=9, frameon=False, ncol=1)

plt.tight_layout()
fig_equidade_path = f'{OUTDIR}/fase5_equidade_auroc.png'
fig.savefig(fig_equidade_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura salva: {fig_equidade_path}')

## 2 · Classification metrics at operational thresholds

Three thresholds: the Youden index (statistical reference), and two
clinical thresholds (≈0.10 and ≈0.20) within the positive net-benefit range already mapped in Phase 4's
decision curve. Accuracy and F1 are reported for completeness, with the caveat already noted that they
are uninformative at this prevalence; MCC is the preferred scalar summary.

In [ ]:
def limiar_youden(y, p):
    fpr, tpr, limiares = roc_curve(y, p)
    j = tpr - fpr
    return limiares[np.argmax(j)]


LIMIARES_CLINICOS = [0.10, 0.20]

linhas_classificacao = []
for nome_set, p_teste in predicoes.items():
    limiar_y = limiar_youden(y_teste, p_teste)
    limiares_a_testar = [('Youden', limiar_y)] + [(f'{l:.2f}', l) for l in LIMIARES_CLINICOS]

    for nome_limiar, limiar in limiares_a_testar:
        pred = (p_teste >= limiar).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_teste, pred).ravel()
        sens = tp / (tp + fn) if (tp + fn) else np.nan
        espec = tn / (tn + fp) if (tn + fp) else np.nan
        vpp = tp / (tp + fp) if (tp + fp) else np.nan
        vpn = tn / (tn + fn) if (tn + fn) else np.nan
        linhas_classificacao.append({
            'set': nome_set, 'limiar_nome': nome_limiar, 'limiar_valor': limiar,
            'VP': tp, 'FP': fp, 'VN': tn, 'FN': fn,
            'sensibilidade': sens, 'especificidade': espec, 'VPP': vpp, 'VPN': vpn,
            'accuracy': accuracy_score(y_teste, pred), 'F1': f1_score(y_teste, pred),
            'MCC': matthews_corrcoef(y_teste, pred),
        })

tab_classificacao = pd.DataFrame(linhas_classificacao)
classif_path = f'{OUTDIR}/modelo_A_fase5_classificacao.csv'
tab_classificacao.to_csv(classif_path, index=False, encoding='utf-8-sig')
print(f'Classification table saved: {classif_path}')
print(tab_classificacao.to_string(index=False))
print()
print('Note: accuracy and F1 are uninformative at this prevalence (~15% deaths) — see Chicco & '
      'Jurman (2020). MCC is the preferred scalar summary.')

### Note — reading the classification table, practical use of thresholds, and the transfer-bias caveat

**What the table shows.** The model produces, for each patient, a continuous probability of death — not
a Discharge/Death decision. The threshold is the yardstick applied to that probability to turn it into a
binary action (e.g., `probability ≥ threshold → flagged as high risk`). Each table row combines a
variable set with one of these thresholds, with TP, FP, TN, and FN from the confusion matrix at that
cut, and sensitivity/specificity/PPV/NPV derived from it.

**The three thresholds, and where they come from:**
- **Youden** — computed from each model's ROC curve, maximizing sensitivity + specificity − 1 (Youden,
  1950). It varies by set (0.139 in Set 1, 0.103 in Set 2), because it depends on how that specific
  model distributes its probabilities.
- **0.10 and 0.20** — fixed values, the same for both sets, chosen within the positive net-benefit range
  of the decision curve (Phase 4): 0.10 as a high-sensitivity cut, 0.20 as a higher-specificity cut.

**Where the threshold would enter practice.** In a prospective risk calculator (Phase 6), the output
would not be just "estimated probability: 23%" — it would be "23% → above the alert threshold (20%)".
It is this crossing that would decide whether the patient enters a priority list, triggers a
notification, or any operational action tied to the tool. The threshold choice depends on the cost of
the triggered action: a **lower threshold (e.g., 0.10)** makes sense when the action is cheap (review
the chart more closely) and the cost of missing a true case is high; a **higher threshold (e.g., 0.20)**
makes sense when the action has real cost (activating a rapid response team, reserving a bed), and it is
worth being more selective even at the price of sensitivity.

**Accuracy and F1** are reported for completeness but are uninformative at this ~15% mortality
prevalence: a model that always predicts "Discharge" would already have high accuracy, and F1 ignores
true negatives.

**MCC** is the preferred scalar summary: it ranges from −1 to +1, uses all four confusion-matrix cells
simultaneously, and is invariant to swapping the "positive" class — more robust to imbalance than
accuracy and F1 (Chicco & Jurman, 2020). In this table, Set 2's MCC surpasses Set 1's at every
threshold, confirming from this angle the AUROC advantage observed since Phase 4.

**Caveat — transfer bias.** These results use the primary definition of `mortalidade_hospitalar`, in
which patients transferred to another hospital before death are coded as survivors (Transfer Bias Note).
This may slightly inflate apparent specificity, since part of the "correct" survivor classifications
actually conceals unknown real outcomes. Section 7 (adjusted definition) will show whether the hierarchy
between sets and the threshold choice hold under the definition that excludes transferred patients.

## 3 · Post-hoc recalibration — bootstrap decision and *out-of-fold* fitting

Three steps:

1. **Decision**: 95% CI (bootstrap, 1,000 resamples) of the calibration intercept and slope on the test
   set. If the intercept's CI excludes 0, or the slope's excludes 1, there is evidence of systematic
   deviation.
2. **Fitting**: if recalibration is indicated, Platt scaling is fitted on *out-of-fold* training
   predictions (`StratifiedGroupKFold`, 5 folds, the same used for Phase 4's hyperparameter tuning) —
   never on the test set.
3. **Application**: the resulting transformation is applied to the test predictions, and the Brier score
   before/after is compared.

In [ ]:
def inclinacao_intercepto_calibracao(y, p, eps=1e-6):
    p_clip = np.clip(p, eps, 1 - eps)
    logit_p = np.log(p_clip / (1 - p_clip)).reshape(-1, 1)
    modelo_cal = LogisticRegression(penalty=None, solver='lbfgs')
    modelo_cal.fit(logit_p, y)
    return modelo_cal.intercept_[0], modelo_cal.coef_[0][0]


def bootstrap_ic_calibracao(y, p, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y)
    interceptos, inclinacoes = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y[idx])) < 2:
            continue
        i, s = inclinacao_intercepto_calibracao(y[idx], p[idx])
        interceptos.append(i); inclinacoes.append(s)
    ic_intercepto = (np.percentile(interceptos, 2.5), np.percentile(interceptos, 97.5))
    ic_inclinacao = (np.percentile(inclinacoes, 2.5), np.percentile(inclinacoes, 97.5))
    return ic_intercepto, ic_inclinacao


linhas_decisao_recal = []
for nome_set, p_teste in predicoes.items():
    intercepto, inclinacao = inclinacao_intercepto_calibracao(y_teste, p_teste)
    ic_intercepto, ic_inclinacao = bootstrap_ic_calibracao(y_teste, p_teste)
    recalibrar = (ic_intercepto[0] > 0 or ic_intercepto[1] < 0) or (ic_inclinacao[0] > 1 or ic_inclinacao[1] < 1)
    linhas_decisao_recal.append({
        'set': nome_set, 'intercepto': intercepto, 'inclinacao': inclinacao,
        'IC_intercepto_low': ic_intercepto[0], 'IC_intercepto_high': ic_intercepto[1],
        'IC_inclinacao_low': ic_inclinacao[0], 'IC_inclinacao_high': ic_inclinacao[1],
        'recalibrar': recalibrar,
    })
    print(f"{nome_set}: intercepto={intercepto:.3f} IC95%[{ic_intercepto[0]:.3f}, {ic_intercepto[1]:.3f}] | "
          f"slope={inclinacao:.3f} 95%CI[{ic_inclinacao[0]:.3f}, {ic_inclinacao[1]:.3f}] | "
          f"recalibrar = {recalibrar}")

tab_decisao_recal = pd.DataFrame(linhas_decisao_recal)

In [ ]:
def obter_predicoes_out_of_fold(modelo_ajustado, Xtr, ytr, groups_tr, cats, n_splits=5, seed=SEED):
    # Out-of-fold predictions on training, refitting a clone of the model (same hyperparameters) on
    # fold of the StratifiedGroupKFold — never using the test set itself.
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    p_oof = np.full(len(ytr), np.nan)
    is_lightgbm = 'LGBM' in type(modelo_ajustado).__name__

    for idx_treino_fold, idx_holdout_fold in cv.split(Xtr, ytr, groups=groups_tr):
        modelo_fold = clone(modelo_ajustado)
        X_tr_fold = Xtr.iloc[idx_treino_fold]
        X_ho_fold = Xtr.iloc[idx_holdout_fold]
        y_tr_fold = ytr[idx_treino_fold]

        if is_lightgbm:
            modelo_fold.fit(X_tr_fold, y_tr_fold, categorical_feature=cats)
        else:
            modelo_fold.fit(X_tr_fold, y_tr_fold)

        p_oof[idx_holdout_fold] = modelo_fold.predict_proba(X_ho_fold)[:, 1]

    return p_oof


linhas_recalibracao = []
predicoes_recalibradas = {}
for nome_set, p_teste in predicoes.items():
    linha_decisao = tab_decisao_recal[tab_decisao_recal['set'] == nome_set].iloc[0]
    if not linha_decisao['recalibrar']:
        print(f'{nome_set}: 95% CI compatible with adequate calibration — recalibration not applied.')
        predicoes_recalibradas[nome_set] = p_teste
        continue

    Xtr = dados_prep[nome_set]['Xtr']
    cats = dados_prep[nome_set]['cats']
    ytr = treino_t[ALVO].values
    groups_tr = treino_t[GROUP_COL].values
    modelo = modelos[nome_set]

    p_oof = obter_predicoes_out_of_fold(modelo, Xtr, ytr, groups_tr, cats)
    intercepto_platt, inclinacao_platt = inclinacao_intercepto_calibracao(ytr, p_oof)

    logit_teste = np.log(np.clip(p_teste, 1e-6, 1 - 1e-6) / (1 - np.clip(p_teste, 1e-6, 1 - 1e-6)))
    logit_recal = intercepto_platt + inclinacao_platt * logit_teste
    p_recal = 1 / (1 + np.exp(-logit_recal))
    predicoes_recalibradas[nome_set] = p_recal

    brier_antes = brier_score_loss(y_teste, p_teste)
    brier_depois = brier_score_loss(y_teste, p_recal)
    intercepto_pos, inclinacao_pos = inclinacao_intercepto_calibracao(y_teste, p_recal)

    linhas_recalibracao.append({
        'set': nome_set, 'intercepto_platt_oof': intercepto_platt,
        'inclinacao_platt_oof': inclinacao_platt, 'Brier_antes': brier_antes, 'Brier_depois': brier_depois,
        'intercepto_pos_recal': intercepto_pos, 'inclinacao_pos_recal': inclinacao_pos,
    })
    print(f'{nome_set}: recalibrado (Platt out-of-fold). Brier {brier_antes:.4f} -> {brier_depois:.4f}')

tab_recalibracao = pd.DataFrame(linhas_recalibracao)
recal_path = f'{OUTDIR}/modelo_A_fase5_recalibracao.csv'
decisao_path = f'{OUTDIR}/modelo_A_fase5_decisao_recalibracao.csv'
tab_recalibracao.to_csv(recal_path, index=False, encoding='utf-8-sig')
tab_decisao_recal.to_csv(decisao_path, index=False, encoding='utf-8-sig')
print(f'Tabelas salvas: {recal_path} | {decisao_path}')

In [ ]:
# --- Plot: bootstrap distribution of the intercept (decision) + calibration curve before/after ---
fig, axes = plt.subplots(2, len(predicoes), figsize=(7 * len(predicoes), 10))
if len(predicoes) == 1:
    axes = axes.reshape(2, 1)

rng_plot = np.random.default_rng(SEED)

for col_idx, (nome_set, p_teste) in enumerate(predicoes.items()):
    linha_decisao = tab_decisao_recal[tab_decisao_recal['set'] == nome_set].iloc[0]

    # --- Row 1: bootstrap distribution of the intercept, with 95% CI and the reference at 0 ---
    n = len(y_teste)
    interceptos_boot = []
    for _ in range(N_BOOT):
        idx = rng_plot.integers(0, n, n)
        if len(np.unique(y_teste[idx])) < 2:
            continue
        i_boot, _ = inclinacao_intercepto_calibracao(y_teste[idx], p_teste[idx])
        interceptos_boot.append(i_boot)

    eixo_top = axes[0, col_idx]
    eixo_top.hist(interceptos_boot, bins=40, color='#D9D9D9', edgecolor='white', zorder=1)
    eixo_top.axvline(0, color='gray', linestyle=':', linewidth=2,
                      label='Reference (perfect calibration) = 0', zorder=2)
    eixo_top.axvline(linha_decisao['intercepto'], color='#C00000', linestyle='-', linewidth=2.5,
                      label=f"Intercepto observado = {linha_decisao['intercepto']:.3f}", zorder=3)
    eixo_top.axvline(linha_decisao['IC_intercepto_low'], color='#1F3864', linestyle='--', linewidth=1.5, zorder=2)
    eixo_top.axvline(linha_decisao['IC_intercepto_high'], color='#1F3864', linestyle='--', linewidth=1.5,
                      label='IC 95% (bootstrap)', zorder=2)
    eixo_top.set_title(f'{nome_set}\nRecalibrar = {linha_decisao["recalibrar"]}')
    eixo_top.set_xlabel('Calibration intercept (bootstrap)')
    eixo_top.legend(fontsize=8, loc='upper left')

    # --- Row 2: calibration curve before vs. after recalibration ---
    p_recal = predicoes_recalibradas[nome_set]
    identicas = np.allclose(p_teste, p_recal)

    frac_antes, media_antes = calibration_curve(y_teste, p_teste, n_bins=10, strategy='quantile')
    frac_depois, media_depois = calibration_curve(y_teste, p_recal, n_bins=10, strategy='quantile')

    eixo_bot = axes[1, col_idx]
    eixo_bot.plot(media_antes, frac_antes, 'o-', color='#C00000', linewidth=2.5, markersize=8,
                  label='Before recalibration', zorder=2)
    eixo_bot.plot(media_depois, frac_depois, 's--', color='#1F3864', linewidth=1.5, markersize=5,
                  label='After recalibration', zorder=3)
    eixo_bot.plot([0, 1], [0, 1], '--', color='gray', linewidth=1, zorder=1)
    eixo_bot.set_xlabel('Probabilidade prevista')
    eixo_bot.set_ylabel('Observed frequency')
    if identicas:
        eixo_bot.text(0.05, 0.90, 'Identical curves — recalibration not applied',
                      transform=eixo_bot.transAxes, fontsize=8, color='#C00000')
    eixo_bot.legend(fontsize=8, loc='lower right')

plt.tight_layout()
fig_recal_path = f'{OUTDIR}/fase5_recalibracao_diagnostico.png'
fig.savefig(fig_recal_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura salva: {fig_recal_path}')

### Note — why recompute calibration, if Phase 4 already did it?

Phase 4 computed the calibration slope and intercept **once on the test set**, as a descriptive
diagnostic: a snapshot of the model's state, with no associated correction decision. Here the same
metric is recomputed, but with a different purpose — deciding whether recalibration is worthwhile, and
correcting in a technically correct way if the decision is yes.

**What changes.** The code resamples the test set 1,000 times (bootstrap) and recomputes the
intercept/slope at each resample, building a 95% confidence interval. This turns the question from "what
is the observed value?" into "is this deviation statistically reliable, or could it be just sampling
noise?". The decision rule only triggers recalibration if the intercept's CI does not cover 0, or the
slope's does not cover 1.

**Why this matters.** Without this criterion, an isolated deviation (like the −0.14 intercept seen in
Phase 4, Set 1) could lead to correcting something that, given the test sample size, is indistinguishable
from perfect calibration — reintroducing variation where there was no real problem.

**Result in this case:** in both sets, the CIs cover the reference values (0 and 1) —
`recalibrar = False`. The predictions carried through the rest of Phase 5 remain the same as Phase 4's,
without transformation.

**If the decision had been to recalibrate.** The correction (next cell, Platt scaling) would be fitted
on *out-of-fold* training predictions — never on the test set, unlike Phase 4's descriptive reading.
This avoids reintroducing optimism: computing the deviation and correcting it on the same test set would
artificially inflate the apparent quality of calibration (Van Calster et al., 2016).

## 4 · Transparency — local SHAP

Case-level explanations for six illustrative patients of the temporal test, per variable set: two true
positives, two true negatives, one false positive, and one false negative — classified by the Youden
threshold already computed in Section 2. Reuses `TreeExplainer` directly on the already-loaded models,
with no refitting.

In [ ]:
import shap

CASOS_POR_CATEGORIA = {'VP': 2, 'VN': 2, 'FP': 1, 'FN': 1}

def selecionar_casos_ilustrativos(y, p, limiar, casos_por_categoria=CASOS_POR_CATEGORIA, seed=SEED):
    rng = np.random.default_rng(seed)
    pred = (p >= limiar).astype(int)
    categorias = {
        'VP': np.where((y == 1) & (pred == 1))[0],
        'VN': np.where((y == 0) & (pred == 0))[0],
        'FP': np.where((y == 0) & (pred == 1))[0],
        'FN': np.where((y == 1) & (pred == 0))[0],
    }
    selecionados = {}
    for nome_cat, n in casos_por_categoria.items():
        disponiveis = categorias[nome_cat]
        n_efetivo = min(n, len(disponiveis))
        selecionados[nome_cat] = list(rng.choice(disponiveis, size=n_efetivo, replace=False)) if n_efetivo else []
    return selecionados


for nome_set, p_teste in predicoes.items():
    Xte = dados_prep[nome_set]['Xte']
    modelo = modelos[nome_set]
    limiar_y = tab_classificacao[(tab_classificacao['set'] == nome_set) &
                                  (tab_classificacao['limiar_nome'] == 'Youden')]['limiar_valor'].iloc[0]

    explainer = shap.TreeExplainer(modelo)
    valores_shap = explainer(Xte)  # shap.Explanation object, for plots.waterfall

    casos = selecionar_casos_ilustrativos(y_teste, p_teste, limiar_y)
    sufixo = nome_set.split()[1].strip('()')

    for nome_cat, indices in casos.items():
        for i, idx_local in enumerate(indices):
            plt.figure()
            shap.plots.waterfall(valores_shap[idx_local], show=False, max_display=15)
            nome_arquivo = f'fase5_shap_local_{sufixo}_{nome_cat}_{i+1}.png'
            plt.savefig(f'{OUTDIR}/{nome_arquivo}', dpi=150, bbox_inches='tight')
            plt.close()
            print(f'  {nome_set} · {nome_cat} #{i+1}: figura salva ({nome_arquivo})')

print('Local (SHAP) explanations saved for both sets.')

In [ ]:
# --- Single preview figure: few patients, to visualize the shape of local SHAP ---
from matplotlib.patches import Patch

NOME_CONJUNTO_PREVIA = 'Conjunto 2 (brutos)'  # switch to 'Conjunto 1 (escores)' to see the other set
N_PREVIA = 3
N_FEATURES_PREVIA = 8

p_teste = predicoes[NOME_CONJUNTO_PREVIA]
Xte = dados_prep[NOME_CONJUNTO_PREVIA]['Xte']
modelo = modelos[NOME_CONJUNTO_PREVIA]
limiar_y = tab_classificacao[(tab_classificacao['set'] == NOME_CONJUNTO_PREVIA) &
                              (tab_classificacao['limiar_nome'] == 'Youden')]['limiar_valor'].iloc[0]

pred = (p_teste >= limiar_y).astype(int)
candidatos_vp = np.where((y_teste == 1) & (pred == 1))[0]
candidatos_vn = np.where((y_teste == 0) & (pred == 0))[0]
candidatos_fp = np.where((y_teste == 0) & (pred == 1))[0]

rng_previa = np.random.default_rng(SEED)
indices_previa, rotulos_previa = [], []
for nome_cat, candidatos in [('VP', candidatos_vp), ('VN', candidatos_vn), ('FP', candidatos_fp)]:
    if len(candidatos) and len(indices_previa) < N_PREVIA:
        indices_previa.append(int(rng_previa.choice(candidatos)))
        rotulos_previa.append(nome_cat)

explainer = shap.TreeExplainer(modelo)
valores_shap = explainer(Xte)

fig, axes = plt.subplots(1, len(indices_previa), figsize=(6 * len(indices_previa), 5.5))
if len(indices_previa) == 1:
    axes = [axes]

for eixo, idx_local, nome_cat in zip(axes, indices_previa, rotulos_previa):
    contrib = pd.Series(valores_shap.values[idx_local], index=Xte.columns)
    top = contrib.reindex(contrib.abs().sort_values(ascending=False).index[:N_FEATURES_PREVIA]).iloc[::-1]
    cores = ['#C00000' if v > 0 else '#1F3864' for v in top.values]
    eixo.barh(top.index, top.values, color=cores)
    eixo.axvline(0, color='black', linewidth=0.8)
    eixo.set_title(f'{nome_cat} — p previsto = {p_teste[idx_local]:.2f}')
    eixo.set_xlabel('SHAP contribution (logit)')

legenda_cores = [
    Patch(facecolor='#C00000', label='Aumenta o risco previsto'),
    Patch(facecolor='#1F3864', label='Reduz o risco previsto'),
]
fig.legend(handles=legenda_cores, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.04), frameon=False)

plt.suptitle(f'Preview — {NOME_CONJUNTO_PREVIA}  (Youden threshold = {limiar_y:.3f})', y=1.03)
plt.tight_layout()
fig_previa_path = f'{OUTDIR}/fase5_shap_local_previa.png'
fig.savefig(fig_previa_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Preview figure saved: {fig_previa_path}')

## 5 · TRIPOD-AI reporting and risk-of-bias assessment (PROBAST)

The full checklist is a textual, not computational, document — it will be consolidated in Phase 6's final
report. This cell merely organizes, in a table, which items already have evidence produced across Phases
1–5, to facilitate later writing.

In [ ]:
checklist_tripod_probast = pd.DataFrame([
    {'item': 'Fonte e elegibilidade dos dados',            'dominio': 'TRIPOD-AI', 'evidencia_fase': 'Fase 1',
     'status': 'Completed'},
    {'item': 'Outcome definition and temporal contract',  'dominio': 'TRIPOD-AI', 'evidencia_fase': 'Fase 1',
     'status': 'Completed'},
    {'item': 'Table 1 — cohort characteristics',       'dominio': 'TRIPOD-AI', 'evidencia_fase': 'Fase 2',
     'status': 'Completed'},
    {'item': 'Validation strategy (temporal + random)', 'dominio': 'TRIPOD-AI/PROBAST', 'evidencia_fase': 'Fase 3',
     'status': 'Completed'},
    {'item': 'Comparison with clinical reference (SAPS3)',  'dominio': 'TRIPOD-AI', 'evidencia_fase': 'Fases 3-4',
     'status': 'Completed'},
    {'item': 'Discrimination (AUROC/AUPRC) and DeLong test', 'dominio': 'TRIPOD-AI', 'evidencia_fase': 'Fase 4',
     'status': 'Completed'},
    {'item': 'Calibration (curve, slope, intercept, Brier)', 'dominio': 'TRIPOD-AI', 'evidencia_fase': 'Fase 4',
     'status': 'Completed'},
    {'item': 'Clinical utility (decision curve)',       'dominio': 'TRIPOD-AI', 'evidencia_fase': 'Fase 4',
     'status': 'Completed'},
    {'item': 'Transparency — global importance (SHAP)',  'dominio': 'TRIPOD-AI', 'evidencia_fase': 'Fase 4',
     'status': 'Completed'},
    {'item': 'Classification metrics at operational thresholds', 'dominio': 'TRIPOD-AI', 'evidencia_fase': 'Fase 5',
     'status': 'Completed'},
    {'item': 'Equidade — desempenho por subgrupo',         'dominio': 'PROBAST (aplicabilidade)', 'evidencia_fase': 'Fase 5',
     'status': 'Completed'},
    {'item': 'Post-hoc recalibration (if indicated)',         'dominio': 'TRIPOD-AI', 'evidencia_fase': 'Fase 5',
     'status': 'Completed'},
    {'item': 'Transparency — local explanations (SHAP)',  'dominio': 'TRIPOD-AI', 'evidencia_fase': 'Fase 5',
     'status': 'Completed'},
    {'item': 'Sensitivity — non-index admissions (GroupKFold)', 'dominio': 'PROBAST (risk of bias)', 'evidencia_fase': 'Fase 5',
     'status': 'Completed'},
    {'item': 'Transfer bias — quantification (adjusted definition, DeLong)', 'dominio': 'PROBAST (risk of bias — outcome)',
     'evidencia_fase': 'Fase 5', 'status': 'Completed'},
    {'item': 'Global risk of bias (PROBAST — 4 domains)', 'dominio': 'PROBAST', 'evidencia_fase': 'Fase 6',
     'status': 'Pending — write-up'},
    {'item': 'Manuscript writing per TRIPOD-AI (27-item checklist)', 'dominio': 'TRIPOD-AI',
     'evidencia_fase': 'Fase 6', 'status': 'Pending — write-up'},
])

checklist_path = f'{OUTDIR}/modelo_A_fase5_checklist_tripod_probast.csv'
checklist_tripod_probast.to_csv(checklist_path, index=False, encoding='utf-8-sig')
print(f'Checklist salvo: {checklist_path}')
print(checklist_tripod_probast.to_string(index=False))

## 6 · Robustness — re-including repeated (non-index) admissions

Model A's main cohort uses the index admission (one row per patient). As a robustness check, this section
reincorporates repeated (non-index) admissions, with the unit of analysis becoming the admission —
requiring cross-validation grouped by patient (`GroupKFold`), since multiple admissions of the same
patient are no longer independent.

**File origin**: `modelo_A_todas_internacoes.csv` is generated by the
`02_sensitivity_all_admissions` notebook (Phase 1b), which generalizes the identity resolution and
sheet integration validated in Phases 1 and 2 to all admissions (not only the index admission).

In [ ]:
FASE_ROBUSTEZ_PATH = 'outputs/phase1b/modelo_A_todas_internacoes.csv'  # generated by notebook 02 (Phase 1b)

if not os.path.exists(FASE_ROBUSTEZ_PATH):
    print(f'File not found: {FASE_ROBUSTEZ_PATH}')
    print('Run the Phase 1b notebook (02_sensitivity_all_admissions.ipynb) before continuing with this section.')
else:
    df_todas = pd.read_csv(FASE_ROBUSTEZ_PATH)
    if 'ano' not in df_todas.columns:
        df_todas['ano'] = pd.to_datetime(df_todas['Month Year']).dt.year

    treino_robustez = df_todas[df_todas['ano'] <  CORTE_TEMPORAL].copy()
    teste_robustez  = df_todas[df_todas['ano'] >= CORTE_TEMPORAL].copy()
    treino_robustez, teste_robustez = agrupar_diagnostico_principal(treino_robustez, teste_robustez)

    print(f'Robustness (repeated admissions) — training: {len(treino_robustez)} admissions '
          f'({treino_robustez["patient_id"].nunique()} patients) | test: {len(teste_robustez)} '
          f'admissions ({teste_robustez["patient_id"].nunique()} patients)')

    linhas_robustez = []
    for nome_set, nome_motor in melhor_por_set.items():
        Xtr_r, Xte_r, cats_r, nums_r = preparar_boosting(CONJUNTOS_VAR[nome_set], treino_robustez, teste_robustez)
        ytr_r = treino_robustez[ALVO].values
        yte_r = teste_robustez[ALVO].values
        groups_r = treino_robustez['patient_id'].values

        # Cross-validation grouped by patient (GroupKFold): used here only to check
        # AUROC stability across folds — not for hyperparameter tuning nor for generating
        # test predictions, which come from the single fit below (the same winning hyperparameters
        # from Phase 4, no new search).
        cv_robustez = GroupKFold(n_splits=5)
        auroc_cv_dobras = []
        for idx_tr_fold, idx_ho_fold in cv_robustez.split(Xtr_r, ytr_r, groups=groups_r):
            modelo_fold = clone(modelos[nome_set])
            X_tr_fold, X_ho_fold = Xtr_r.iloc[idx_tr_fold], Xtr_r.iloc[idx_ho_fold]
            y_tr_fold, y_ho_fold = ytr_r[idx_tr_fold], ytr_r[idx_ho_fold]
            if 'LGBM' in type(modelo_fold).__name__:
                modelo_fold.fit(X_tr_fold, y_tr_fold, categorical_feature=cats_r)
            else:
                modelo_fold.fit(X_tr_fold, y_tr_fold)
            p_ho_fold = modelo_fold.predict_proba(X_ho_fold)[:, 1]
            if len(np.unique(y_ho_fold)) > 1:
                auroc_cv_dobras.append(roc_auc_score(y_ho_fold, p_ho_fold))

        modelo_base = clone(modelos[nome_set])
        if 'LGBM' in type(modelo_base).__name__:
            modelo_base.fit(Xtr_r, ytr_r, categorical_feature=cats_r)
        else:
            modelo_base.fit(Xtr_r, ytr_r)

        p_teste_r = modelo_base.predict_proba(Xte_r)[:, 1]
        linhas_robustez.append({
            'set': nome_set, 'motor': nome_motor,
            'AUROC_indice': roc_auc_score(y_teste, predicoes[nome_set]),
            'AUROC_todas_internacoes': roc_auc_score(yte_r, p_teste_r),
            'AUROC_cv_media_dobras': np.mean(auroc_cv_dobras) if auroc_cv_dobras else np.nan,
            'AUROC_cv_desvio_dobras': np.std(auroc_cv_dobras) if auroc_cv_dobras else np.nan,
            'Brier_indice': brier_score_loss(y_teste, predicoes[nome_set]),
            'Brier_todas_internacoes': brier_score_loss(yte_r, p_teste_r),
            'n_teste_indice': len(y_teste), 'n_teste_todas': len(yte_r),
        })

    tab_robustez = pd.DataFrame(linhas_robustez)
    robustez_path = f'{OUTDIR}/modelo_A_fase5_robustez_internacoes_repetidas.csv'
    tab_robustez.to_csv(robustez_path, index=False, encoding='utf-8-sig')
    print(f'Tabela de robustez salva: {robustez_path}')
    print(tab_robustez.to_string(index=False))

### Note — reading the robustness table (repeated admissions)

**What the table compares.** This section tests whether Model A's results hold when all admissions in the
database are included (not only the index admission used in the original training). Three relevant
comparisons:

**1. Index AUROC vs. all-admissions AUROC.** AUROC drops slightly in the expanded cohort in both sets
(Set 1: 0.845→0.821; Set 2: 0.907→0.880) — a small and expected drop, since the expanded cohort is more
heterogeneous (it includes readmissions, with clinical dynamics different from the first admission) and
the model was fitted only on index admissions. The reading is positive: the model still discriminates
well (AUROC > 0.80) outside the scenario it was built in.

**2. Index Brier vs. all-admissions Brier.** The Brier score (calibration error; lower is better) worsens
in the expanded cohort, in the same direction as the AUROC drop (Set 1: 0.083→0.100; Set 2:
0.063→0.084) — consistent with that cohort's greater heterogeneity and with calibration having been
fitted only on index admissions.

**3. Test AUROC vs. CV AUROC (fold mean).** `AUROC_todas_internacoes` is a single number: the model is
trained once and evaluated once on the test set (years ≥2023). `AUROC_cv_media_dobras` comes from
cross-validation grouped by patient (`GroupKFold`, 5 folds) **within training** — training is split into
5 parts, each serving as a temporary validation while the other four train, repeated 5 times.
`GroupKFold` guarantees that all admissions of the same patient always stay on the same side of the
split, avoiding leakage.

CV's purpose is not to replace the test set, but to check **stability**: a single test number may be
good or bad by chance, depending on which specific patients fell into that split. The low standard
deviation across folds (Set 1: 0.011; Set 2: 0.008) indicates that model performance is consistent across
different training subgroups of patients, with no discrepant fold.

**Conclusion of this section.** The model maintains satisfactory discrimination and stable behavior when
extrapolated to the all-admissions cohort, with a modest and expected drop in AUROC and Brier relative to
the original training cohort — no evidence that the model depends critically on any peculiarity unique
to the index admission.

### Note — ablation curve and simplified score (original cohort)

Previous sections evaluated Model A with the full variable set. This section tests whether a smaller
subset, selected by mean SHAP importance (mean |SHAP| on training), sustains equivalent discrimination —
yielding a simpler score, with fewer variables to collect in clinical use.

The ablation curve below refits Set 2's (raw) winning engine for different subset sizes (top-N by
importance), plotting each one's AUROC against the full model's AUROC. The next two cells directly
compare the full model with an N=30-variable score (ROC curve) and test, by DeLong, the smallest N
statistically equivalent to the full model.

In [ ]:
# --- Ablation curve: AUROC vs. number of variables (ranking by mean |SHAP|) ---
from sklearn.base import clone

NOME_CONJUNTO = 'Conjunto 2 (brutos)'  # adjust to the set of interest

Xtr = dados_prep[NOME_CONJUNTO]['Xtr']
Xte = dados_prep[NOME_CONJUNTO]['Xte']
cats = dados_prep[NOME_CONJUNTO]['cats']
ytr = treino_t[ALVO].values
modelo_base = modelos[NOME_CONJUNTO]

# Full-model reference (predictions already generated in Section 0) — computed here, without depending
# on the next cell (simplified-score N decision section), which uses the same variables
# but runs after this one in notebook order.
n_variaveis_completo = Xtr.shape[1]
auroc_completo = roc_auc_score(y_teste, predicoes[NOME_CONJUNTO])

TAMANHOS_N = [5, 8, 10, 15, 20, 30, 50, 75, 100, n_variaveis_completo]  # includes the full model as reference

# Importance ranking — reuses the mean |SHAP| already computed (valores_shap from Section 4)
explainer = shap.TreeExplainer(modelo_base)
shap_vals_treino = explainer(Xtr)
importancia_media = pd.Series(np.abs(shap_vals_treino.values).mean(axis=0), index=Xtr.columns)
ranking_variaveis = importancia_media.sort_values(ascending=False).index.tolist()

resultados_ablacao = []
for n in TAMANHOS_N:
    n = min(n, len(ranking_variaveis))
    variaveis_n = ranking_variaveis[:n]
    cats_n = [c for c in cats if c in variaveis_n]

    Xtr_n = Xtr[variaveis_n]
    Xte_n = Xte[variaveis_n]

    modelo_n = clone(modelo_base)
    if 'LGBM' in type(modelo_n).__name__:
        modelo_n.fit(Xtr_n, ytr, categorical_feature=cats_n)
    else:
        modelo_n.fit(Xtr_n, ytr)

    p_n = modelo_n.predict_proba(Xte_n)[:, 1]
    resultados_ablacao.append({'n_variaveis': n, 'AUROC': roc_auc_score(y_teste, p_n)})
    print(f'n={n:>3d}  AUROC={resultados_ablacao[-1]["AUROC"]:.4f}')

tab_ablacao = pd.DataFrame(resultados_ablacao).drop_duplicates(subset='n_variaveis').sort_values('n_variaveis')

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot(tab_ablacao['n_variaveis'], tab_ablacao['AUROC'], 'o-', color='#1F3864', linewidth=2)
ax.axhline(auroc_completo, color='gray', linestyle='--', linewidth=1,
           label=f'Modelo completo (n={n_variaveis_completo})  AUROC={auroc_completo:.3f}')
ax.set_xlabel('Number of variables (top-N by mean |SHAP|)')
ax.set_ylabel('AUROC (temporal validation)')
ax.set_title(f'Ablation curve — {NOME_CONJUNTO}')
ax.legend(fontsize=9)

plt.tight_layout()
fig_ablacao_path = f'{OUTDIR}/fase6_curva_ablacao_{NOME_CONJUNTO.split()[1].strip("()")}.png'
fig.savefig(fig_ablacao_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura salva: {fig_ablacao_path}')
print(tab_ablacao.to_string(index=False))

In [ ]:
# --- DeLong test helper functions (same as Phase 4, reproduced here) ---
def _compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2


def _fast_delong(preds_sorted_transposed, m):
    # preds_sorted_transposed: shape (k models, n samples), positives first (0:m), negatives after (m:)
    k = preds_sorted_transposed.shape[0]
    n = preds_sorted_transposed.shape[1] - m
    tx = np.empty([k, m])
    ty = np.empty([k, n])
    tz = np.empty([k, m + n])
    for r in range(k):
        tx[r, :] = _compute_midrank(preds_sorted_transposed[r, :m])
        ty[r, :] = _compute_midrank(preds_sorted_transposed[r, m:])
        tz[r, :] = _compute_midrank(preds_sorted_transposed[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov


def teste_delong_duas_curvas(y_true, p1, p2):
    # Compares two correlated ROC curves (same test patients) by DeLong's method.
    # Returns (AUROC_difference, p_value). Defined here because the next cell (N decision of the
    # simplified score, Section 6 · Robustness) depends on it -- only _fast_delong/_compute_midrank
    # (internal use) had been reproduced before.
    order = np.argsort(-y_true)
    y_ord = y_true[order]
    m = int(y_ord.sum())
    preds = np.vstack([p1[order], p2[order]])
    aucs, cov = _fast_delong(preds, m)
    diff = aucs[0] - aucs[1]
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    from scipy import stats as sstats
    z = diff / np.sqrt(var) if var > 0 else np.nan
    p_valor = 2 * (1 - sstats.norm.cdf(abs(z))) if var > 0 else np.nan
    return diff, p_valor


In [ ]:
# --- Prepare the full model and the simplified score, then compare the ROC curves ---
from sklearn.base import clone
from sklearn.metrics import roc_curve

NOME_CONJUNTO = 'Conjunto 2 (brutos)'
N_TOP = 30

Xtr = dados_prep[NOME_CONJUNTO]['Xtr']
Xte = dados_prep[NOME_CONJUNTO]['Xte']
cats = dados_prep[NOME_CONJUNTO]['cats']
ytr = treino_t[ALVO].values
modelo_completo = modelos[NOME_CONJUNTO]

# --- Full model: already fitted in Phase 4/5, just retrieve its predictions ---
p_teste_completo = predicoes[NOME_CONJUNTO]
n_variaveis_completo = Xtr.shape[1]
auroc_completo = roc_auc_score(y_teste, p_teste_completo)

# --- Importance ranking (mean |SHAP| on training) ---
explainer = shap.TreeExplainer(modelo_completo)
shap_vals_treino = explainer(Xtr)
importancia_media = pd.Series(np.abs(shap_vals_treino.values).mean(axis=0), index=Xtr.columns)
ranking_variaveis = importancia_media.sort_values(ascending=False).index.tolist()

# --- Simplified score: top-N variables, same engine/hyperparameters, refitted ---
variaveis_top = ranking_variaveis[:N_TOP]
cats_top = [c for c in cats if c in variaveis_top]

Xtr_reduzido = Xtr[variaveis_top]
Xte_reduzido = Xte[variaveis_top]

modelo_reduzido = clone(modelo_completo)
if 'LGBM' in type(modelo_reduzido).__name__:
    modelo_reduzido.fit(Xtr_reduzido, ytr, categorical_feature=cats_top)
else:
    modelo_reduzido.fit(Xtr_reduzido, ytr)

p_teste_reduzido = modelo_reduzido.predict_proba(Xte_reduzido)[:, 1]
auroc_reduzido = roc_auc_score(y_teste, p_teste_reduzido)

print(f'Modelo completo (n={n_variaveis_completo}):     AUROC = {auroc_completo:.4f}')
print(f'Escore simplificado (n={N_TOP}):                AUROC = {auroc_reduzido:.4f}')
print(f'Simplified score variables: {variaveis_top}')

# --- Plot: ROC of the full model vs. the simplified model ---
fpr_completo, tpr_completo, _ = roc_curve(y_teste, p_teste_completo)
fpr_reduzido, tpr_reduzido, _ = roc_curve(y_teste, p_teste_reduzido)

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot(fpr_completo, tpr_completo, color='#1F3864', linewidth=2.2,
        label=f'Modelo completo (n={n_variaveis_completo})  AUROC={auroc_completo:.3f}')
ax.plot(fpr_reduzido, tpr_reduzido, color='#C00000', linewidth=2.2, linestyle='--',
        label=f'Escore simplificado (n={N_TOP})  AUROC={auroc_reduzido:.3f}')
ax.plot([0, 1], [0, 1], color='gray', linewidth=1, linestyle=':')

ax.set_xlabel('1 − especificidade (FPR)')
ax.set_ylabel('Sensibilidade (TPR)')
ax.set_title('ROC curve — full model vs. simplified score\n(temporal validation, same test set)')
ax.legend(loc='lower right', fontsize=9)

plt.tight_layout()
fig_parsimonia_path = f'{OUTDIR}/fase6_roc_completo_vs_simplificado.png'
fig.savefig(fig_parsimonia_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura salva: {fig_parsimonia_path}')

In [ ]:
# --- N decision of the simplified score: DeLong against the full model, finer grid ---
CANDIDATOS_N = [20, 22, 24, 26, 28, 30]

resultados_candidatos_n = []
for n in CANDIDATOS_N:
    variaveis_n = ranking_variaveis[:n]
    cats_n = [c for c in cats if c in variaveis_n]
    Xtr_n, Xte_n = Xtr[variaveis_n], Xte[variaveis_n]

    modelo_n = clone(modelo_completo)
    if 'LGBM' in type(modelo_n).__name__:
        modelo_n.fit(Xtr_n, ytr, categorical_feature=cats_n)
    else:
        modelo_n.fit(Xtr_n, ytr)

    p_n = modelo_n.predict_proba(Xte_n)[:, 1]
    diff, p_valor = teste_delong_duas_curvas(y_teste, p_n, p_teste_completo)
    resultados_candidatos_n.append({
        'n_variaveis': n,
        'AUROC': roc_auc_score(y_teste, p_n),
        'diferenca_vs_completo': diff,
        'p_valor': p_valor,
        'nao_significativo': p_valor >= 0.05,
    })

tab_candidatos_n = pd.DataFrame(resultados_candidatos_n)
print(tab_candidatos_n.to_string(index=False))

# --- Automatic suggestion: the smallest tested N whose p-value is not significant ---
candidatos_ok = tab_candidatos_n[tab_candidatos_n['nao_significativo']]
if len(candidatos_ok):
    n_sugerido = candidatos_ok['n_variaveis'].min()
    print(f'\nSmallest N with no statistically significant difference vs. the full model: {n_sugerido}')
else:
    print('\nNenhum candidato testado ficou estatisticamente equivalente ao modelo completo — '
          'considere testar valores maiores.')

tab_candidatos_path = f'{OUTDIR}/fase6_candidatos_n_delong.csv'
tab_candidatos_n.to_csv(tab_candidatos_path, index=False, encoding='utf-8-sig')
print(f'Tabela salva: {tab_candidatos_path}')

In [ ]:
# --- Figure: mortality by category — AdmissionReasonName and AdmissionMainDiagnosisName_grp ---
def intervalo_wilson(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = k / n
    denom = 1 + z**2 / n
    centro = (p + z**2 / (2 * n)) / denom
    margem = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denom
    return max(0, centro - margem), min(1, centro + margem)


def tabela_mortalidade_por_categoria(df_in, col, alvo=ALVO, n_min=20):
    linhas = []
    for categoria, sub in df_in.groupby(col):
        n = len(sub)
        if n < n_min:
            continue
        k = int(sub[alvo].sum())
        taxa = k / n
        ic_low, ic_high = intervalo_wilson(k, n)
        linhas.append({'categoria': categoria, 'n': n, 'obitos': k, 'taxa_obito': taxa,
                        'ic_low': ic_low, 'ic_high': ic_high})
    return pd.DataFrame(linhas).sort_values('taxa_obito', ascending=True).reset_index(drop=True)


# AdmissionMainDiagnosisName_grp: same top-20 + "other" grouping, applied to the whole base (descriptive use,
# not the same object used in model training, which is fitted only on each validation's training split)
N_TOP_DIAG_DESCRITIVO = 20
top_categorias_desc = df['AdmissionMainDiagnosisName'].value_counts().nlargest(N_TOP_DIAG_DESCRITIVO).index
df_desc = df.copy()
df_desc['AdmissionMainDiagnosisName_grp_desc'] = df_desc['AdmissionMainDiagnosisName'].where(
    df_desc['AdmissionMainDiagnosisName'].isin(top_categorias_desc), 'outros')

tab_reason = tabela_mortalidade_por_categoria(df, 'AdmissionReasonName', n_min=20)
tab_diag = tabela_mortalidade_por_categoria(df_desc, 'AdmissionMainDiagnosisName_grp_desc', n_min=20)

fig, axes = plt.subplots(1, 2, figsize=(15, max(5, 0.35 * max(len(tab_reason), len(tab_diag)))))

for eixo, tab, titulo in zip(axes, [tab_reason, tab_diag],
                             ['Admission reason (AdmissionReasonName)',
                              'Grouped main diagnosis (top 20 + other)']):
    y_pos = np.arange(len(tab))
    erro = np.vstack([tab['taxa_obito'] - tab['ic_low'], tab['ic_high'] - tab['taxa_obito']])
    eixo.errorbar(tab['taxa_obito'] * 100, y_pos, xerr=erro * 100, fmt='o', color='#1F3864',
                  ecolor='#1F3864', elinewidth=1.3, capsize=3, markersize=5)
    eixo.set_yticks(y_pos)
    eixo.set_yticklabels([f"{cat}  (n={n})" for cat, n in zip(tab['categoria'], tab['n'])], fontsize=8)
    eixo.axvline(df[ALVO].mean() * 100, color='gray', linestyle='--', linewidth=1,
                 label=f'Mortalidade geral = {df[ALVO].mean()*100:.1f}%')
    eixo.set_xlabel('Mortalidade hospitalar (%)')
    eixo.set_title(titulo)
    eixo.legend(fontsize=8, loc='lower right')

plt.tight_layout()
fig_mortalidade_categoria_path = f'{OUTDIR}/fase6_mortalidade_por_categoria.png'
fig.savefig(fig_mortalidade_categoria_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura salva: {fig_mortalidade_categoria_path}')
print(tab_reason.to_string(index=False))
print(tab_diag.to_string(index=False))

In [ ]:
# --- top 10 highest-mortality categories, per variable, in a single panel ---
import matplotlib as mpl

N_TOP_CATEGORIAS = 10

tab_reason_top = tab_reason.sort_values('taxa_obito', ascending=False).head(N_TOP_CATEGORIAS).iloc[::-1].reset_index(drop=True)
tab_diag_top = tab_diag.sort_values('taxa_obito', ascending=False).head(N_TOP_CATEGORIAS).iloc[::-1].reset_index(drop=True)

mortalidade_geral = df[ALVO].mean()
cmap = mpl.colormaps['Reds']
norm = mpl.colors.Normalize(vmin=0, vmax=max(tab_reason_top['taxa_obito'].max(), tab_diag_top['taxa_obito'].max()) * 1.1)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

linha_geral = None  # stores the line handle for the single legend, outside the axes

for eixo, tab, titulo in zip(axes, [tab_reason_top, tab_diag_top],
                             ['Admission reason — top 10 by mortality',
                              'Main diagnosis (grouped) — top 10 by mortality']):
    y_pos = np.arange(len(tab))
    cores = [cmap(norm(v)) for v in tab['taxa_obito']]

    limite_superior = max(tab['ic_high'].max() * 100 * 1.30, mortalidade_geral * 100 * 1.5)
    eixo.set_xlim(0, limite_superior)

    linha = eixo.axvline(mortalidade_geral * 100, color='#1F3864', linestyle='--', linewidth=1.3, zorder=1)
    if linha_geral is None:
        linha_geral = linha  # we only need one handle; the same style repeats in both panels

    barras = eixo.barh(y_pos, tab['taxa_obito'] * 100, color=cores, edgecolor='#333333',
                        linewidth=0.6, height=0.7, zorder=2)

    erro = np.vstack([tab['taxa_obito'] - tab['ic_low'], tab['ic_high'] - tab['taxa_obito']]) * 100
    eixo.errorbar(tab['taxa_obito'] * 100, y_pos, xerr=erro, fmt='none',
                  ecolor='#333333', elinewidth=1, capsize=2, zorder=3)

    for y, (taxa, n, ic_high) in enumerate(zip(tab['taxa_obito'], tab['n'], tab['ic_high'])):
        x_texto = ic_high * 100 + limite_superior * 0.02
        eixo.text(x_texto, y, f'{taxa*100:.1f}%  (n={n})', va='center', fontsize=8,
                  color='#333333', zorder=4)

    eixo.set_yticks(y_pos)
    eixo.set_yticklabels(tab['categoria'], fontsize=9)
    eixo.set_xlabel('Mortalidade hospitalar (%)')
    eixo.set_title(titulo, fontsize=11, fontweight='bold')
    # no per-panel legend — the single legend sits outside the axes, see fig.legend below

    for spine in ['top', 'right']:
        eixo.spines[spine].set_visible(False)
    eixo.grid(axis='x', linestyle=':', alpha=0.4, zorder=0)

plt.suptitle('Categories most associated with hospital death', fontsize=13, fontweight='bold', y=1.04)

# Single legend, shared by both panels, right below the title
fig.legend([linha_geral], [f'Mortalidade geral = {mortalidade_geral*100:.1f}%'],
           loc='upper center', bbox_to_anchor=(0.5, 0.97), ncol=1, fontsize=9, frameon=False)

plt.tight_layout(rect=[0, 0, 1, 0.94])  # reserves space at the top for title + legend

sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar = fig.colorbar(sm, ax=axes, orientation='horizontal', fraction=0.04, pad=0.12, aspect=40)
cbar.set_label('Mortalidade hospitalar (%)', fontsize=9)
cbar.ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1, decimals=0))
fig_top10_path = f'{OUTDIR}/fase6_top10_mortalidade_categorias.png'
fig.savefig(fig_top10_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura salva: {fig_top10_path}')

## Part II — Transfer bias: quantification with the adjusted outcome definition

**Context (Transfer Bias Note).** The source hospital has no hemodynamics service: hemodynamically
unstable patients who need an intervention unavailable locally tend to be transferred to another
hospital — if they do not die before that. `mortalidade_hospitalar` derives exclusively from
`HospitalDischargeName` (Discharge/Death), with no transfer category of its own: every transferred
patient is coded as "Discharge", that is, as a survivor, even without any information on the real
outcome after the transfer. The check already documented showed 589 patients with
`HospitalDestinationName = "Outro hospital"` (Other hospital) in the full raw base (7.8%), with
Cardiovascular concentrating the highest transfer rate (19.4%) and an observed mortality inconsistent
with the expected severity.

**Adjusted definition:** identical to the original, except that patients with
`HospitalDestinationName = "Outro hospital"` now have an unknown hospital outcome — excluded from the
cohort instead of counted as survivors.

**Full reconstruction.** Sections 0–4 below rebuild the index cohort entirely from the raw Excel:
cleaning and `patient_id` via connected components (identical to Phase 1, including recovery of the 555
ambiguous pairs by positional alignment), the two outcome definitions side by side, integration of the
four predictor sheets, and feature engineering with the same logic as Phase 2. Only after that are
Phase 4's two winning engines refitted (same hyperparameters, no new search) under the adjusted
definition, and discrimination (AUROC, DeLong test) compared against the results already obtained with
the original definition.

## 0 · Configuration and loading (Part II)

In [ ]:
import networkx as nx
pd.set_option('display.max_columns', 60)

# === Part II-specific configuration — paths and parameters unique to the reconstruction ===
# CATEGORICAS, ALVO, CORTE_TEMPORAL, SEED, and N_TOP_DIAG were already defined in Part I, with the same values.
FILE_PATH = 'data/Excel01GlobalAdmission.xlsx'
OUTDIR_PARTE2 = 'outputs/phase5_adjusted'
os.makedirs(OUTDIR_PARTE2, exist_ok=True)

COMPARACAO_FASE4_PATH = f'{FASE4_DIR}/modelo_A_fase4_comparacao_completa.csv'
DELONG_FASE4_PATH     = f'{FASE4_DIR}/modelo_A_fase4_delong.csv'

MERGE_KEYS = ['EpimedCode', 'MedicalRecord']
AMBIGUOUS_POLICY = 'exclude'
N_POSICOES = 12

SHEETS = {
    'admissao':      'Folha2',
    'comorbidades':  'Folha3',
    'saps3_razoes':  'Folha4',
    'complic_dia1':  'Folha5',
    'fisiologia6h':  'Folha6',
    'diag_secund':   'Folha8',
}
print('Part II configuration loaded.')


In [ ]:
raw = {nome: pd.read_excel(FILE_PATH, sheet_name=folha, header=1) for nome, folha in SHEETS.items()}
for nome, df_folha in raw.items():
    df_folha['_linha_bruta'] = df_folha.index
    print(f'{nome:14s} ({SHEETS[nome]}): {df_folha.shape[0]:>5} linhas x {df_folha.shape[1]:>3} colunas')

## 1 · Cleaning, patient_id, and index cohort — identical to Phase 1

In [ ]:
BOOL_MAP = {'verdadeiro':1,'falso':0,'sim':1,'no':0,'nao':0,'true':1,'false':0}

def to_bool(series):
    s = series.astype('string').str.strip().str.lower()
    return s.map(BOOL_MAP).astype('Float64')

def clean_booleans(df, protect):
    out = df.copy()
    for col in out.columns:
        if col in protect: continue
        vals = out[col].dropna().astype('string').str.strip().str.lower().unique()
        if len(vals) and set(vals).issubset(BOOL_MAP.keys()):
            out[col] = to_bool(out[col])
    return out

protect = set(MERGE_KEYS) | {'_linha_bruta'}
for nome in raw:
    raw[nome] = clean_booleans(raw[nome], protect)

adm = raw['admissao'].copy()
adm['adm_datetime'] = pd.to_datetime(
    adm['UnitAdmissionDate'].astype(str) + ' ' + adm['Unit AdmissionTime'].astype(str),
    dayfirst=True, errors='coerce')

cols_comparar = [c for c in adm.columns if c != '_linha_bruta']
adm = adm.drop_duplicates(subset=cols_comparar, keep='first').reset_index(drop=True)

def patient_id_linked(df, keys=MERGE_KEYS):
    G = nx.Graph(); G.add_nodes_from(df.index)
    for col in keys:
        for _, idx in df.groupby(col).groups.items():
            idx = list(idx)
            for a in idx[1:]:
                G.add_edge(idx[0], a)
    comp = {n:i for i, c in enumerate(nx.connected_components(G)) for n in c}
    return df.index.map(comp).astype('int64')

adm['patient_id'] = patient_id_linked(adm)

pair_counts = adm.groupby(MERGE_KEYS).size().rename('n_par')
adm = adm.merge(pair_counts, on=MERGE_KEYS, how='left')
adm['par_unico'] = adm['n_par'].eq(1)

adm = adm.sort_values(['patient_id', 'adm_datetime'], kind='stable')
index_cohort = adm.drop_duplicates(subset='patient_id', keep='first').copy()

print(f'Patients (index cohort): {len(index_cohort)} ')

## 2 · Two outcome definitions — original (current Model A) and adjusted (transferred excluded)

`HospitalDischargeName` only distinguishes Discharge/Death — anyone transferred to another hospital
appears as "Discharge" in that column, even without our knowing what happened afterwards.
`HospitalDestinationName`, on the other hand, already distinguishes "Other hospital" from "Home"/other
destinations — that is the information the current Model A definition does not use.

In [ ]:
# --- Original outcome (the same used since Phase 1) ---
def derive_outcome_original(df, col):
    s = df[col].astype('string').str.strip().str.lower()
    return s.map({'death':1,'obito':1,'alta':0}).astype('Float64')

index_cohort['mortalidade_hospitalar_original'] = derive_outcome_original(index_cohort, 'HospitalDischargeName')

# --- Adjusted outcome: patients transferred to another hospital become UNKNOWN outcome ---
index_cohort['transferido_outro_hospital'] = (index_cohort['HospitalDestinationName'] == 'Outro hospital')
index_cohort['mortalidade_hospitalar_ajustada'] = index_cohort['mortalidade_hospitalar_original'].copy()
index_cohort.loc[index_cohort['transferido_outro_hospital'], 'mortalidade_hospitalar_ajustada'] = pd.NA

n_transferidos = int(index_cohort['transferido_outro_hospital'].sum())
print(f'Patients (index cohort) transferred to another hospital: {n_transferidos}')
print(f'Desfecho original conhecido:      {index_cohort["mortalidade_hospitalar_original"].notna().sum()}')
print(f'Desfecho ajustada conhecido: {index_cohort["mortalidade_hospitalar_ajustada"].notna().sum()}')

## 3 · Integration of the four predictor sheets — identical to Phase 1/Phase 1b

In [ ]:
PUBLICO = ['SESA - State Department of Health']
PARTICULAR = ['Particular']

def categorizar_convenio(valor):
    if pd.isna(valor):
        return 'Not reported'
    v = str(valor).strip()
    if v in PUBLICO:
        return 'Public/SUS'
    if v in PARTICULAR:
        return 'Particular'
    return 'Private insurance'

index_cohort['OrigemPagamento'] = index_cohort['HealthInsuranceName'].apply(categorizar_convenio)

FOLHA2_KEEP = [
    'EpimedCode','MedicalRecord','patient_id','_linha_bruta',
    'Age','Gender','Weight','Height','BMI',
    'AdmissionSourceName','AdmissionTypeName','AdmissionReasonName',
    'AdmissionMainDiagnosisName','Month Year','OrigemPagamento',
    'Charlson Comorbidity Index','MFI points','MFI Score','Frail Patient MFI',
    'Saps3Points','Saps3Death Probability Standard Equation','Sofa Score',
    'Status COVID 19',
    'mortalidade_hospitalar_original','mortalidade_hospitalar_ajustada',
    'transferido_outro_hospital','adm_datetime','par_unico',
]
FOLHA2_KEEP = [c for c in FOLHA2_KEEP if c in index_cohort.columns]
base_completa = index_cohort[FOLHA2_KEEP].copy()
PRED_SHEETS = ['comorbidades','saps3_razoes','complic_dia1','fisiologia6h']

clean = base_completa[base_completa['par_unico'].fillna(False)].copy()
for nome in PRED_SHEETS:
    # local variable df_folha (not `df`) — Part I uses `df` for the Phase 2 base, and reusing
    # the same name here would overwrite that global object, even if only after it was last used
    df_folha = raw[nome].drop_duplicates(subset=MERGE_KEYS, keep='first')
    novas = MERGE_KEYS + [c for c in df_folha.columns if c not in clean.columns]
    clean = clean.merge(df_folha[novas], on=MERGE_KEYS, how='left', validate='one_to_one')
clean['integravel'] = True
clean['recuperado_por_posicao'] = False

ambiguas = base_completa[~base_completa['par_unico'].fillna(False)].copy()
recuperados_linhas, falhas_linhas = [], []
for _, adm_row in ambiguas.iterrows():
    linha_alvo = adm_row['_linha_bruta']
    ec, mr = adm_row['EpimedCode'], adm_row['MedicalRecord']
    novos_dados, ok = {}, True
    for nome in PRED_SHEETS:
        df_folha = raw[nome]
        candidatos = df_folha[(df_folha['EpimedCode']==ec) & (df_folha['MedicalRecord']==mr) & (df_folha['_linha_bruta']==linha_alvo)]
        if len(candidatos) != 1:
            ok = False; break
        linha = candidatos.iloc[0]
        for c in df_folha.columns:
            if c not in FOLHA2_KEEP and c != '_linha_bruta':
                novos_dados[c] = linha[c]
    (recuperados_linhas if ok else falhas_linhas).append({**adm_row.to_dict(), **novos_dados} if ok else adm_row.to_dict())

recuperados = pd.DataFrame(recuperados_linhas)
if len(recuperados):
    recuperados['integravel'] = True
    recuperados['recuperado_por_posicao'] = True

base_integrada = pd.concat([clean, recuperados], ignore_index=True)
# Do NOT re-sort by patient_id/adm_datetime here: the official Phase 1 does not re-sort after the concat
# (unique first, recovered at the end). Re-sorting would change the physical row order of
# training and, with it, the result of refitting the gradient boosting engines that use
# stochastic subsampling (Section 6) — even with a fixed random_state and identical data.
# Verified: without this sort, the order exactly reproduces the official modelo_A_fase2.csv's.
print(f'Integrated index cohort: {base_integrada.shape}')

In [ ]:
# --- Check: the difference between the two "known" counts is exactly the number of transferred patients ---
n_original_conhecido = int(index_cohort['mortalidade_hospitalar_original'].notna().sum())
n_ajustada_conhecido = int(index_cohort['mortalidade_hospitalar_ajustada'].notna().sum())

print(f'\nCheck: {n_original_conhecido} (known original) − {n_transferidos} (transferred) '
      f'= {n_original_conhecido - n_transferidos} → matches the known adjusted outcome '
      f'({n_ajustada_conhecido})? {n_original_conhecido - n_transferidos == n_ajustada_conhecido}')

print('\nWhat this confirms: transferred patients are not removed here — their adjusted outcome is only '
      'set to NA. Effective removal by definition happens later, cell by cell, '
      'via .notna() on the relevant outcome column (Sections 5 and 6). This preserves the cohort '
      'original (5,676, same N as Phase 1) intact, since most transferred patients have '
      'HospitalDischargeName = "Alta" (known original outcome, albeit biased).')

In [ ]:
# --- Check: does "known" drop from 5,680 (index_cohort) to 5,676 (base_integrada)? ---
n_original_conhecido_base = int(base_integrada['mortalidade_hospitalar_original'].notna().sum())
n_ajustada_conhecido_base = int(base_integrada['mortalidade_hospitalar_ajustada'].notna().sum())

print(f'Known original outcome (base_integrada, after excluding the 4 ambiguous): {n_original_conhecido_base}')
print(f'Known adjusted outcome (base_integrada, after excluding the 4 ambiguous): {n_ajustada_conhecido_base}')
print(f'\nExpected drop, solely from the 4 ambiguous leaving: 5680 → {5680 - 4} '
      f'(bate com o valor acima? {5680 - 4 == n_original_conhecido_base})')

## 4 · Feature engineering (Phase 2) — identical, no logic changes

In [ ]:
COMORB_GRUPOS = {
    'comorb_cardiovascular': ['IsChfNyhaClass23','IsChfNyhaClass4','IsArterialHypertension',
        'IsAngina','IsPreviousAMI','IsCardiacArrhythmia','IsChronicAtrialFibrilation',
        'IsDeepVenousThrombosis','IsPeripheralArteryDisease'],
    'comorb_renal': ['IsCrfNoDialysis','IsCrfDialysis'],
    'comorb_hepatico': ['IsCirrhosisChildAB','IsCirrhosisChildC','IsHepaticFailure'],
    'comorb_oncologico': ['IsSolidTumorLocoregional','IsSolidTumorMetastatic',
        'IsHematologicalMalignancy','Is Chemotherapy','Is Radiation Therapy'],
    'comorb_respiratorio_cronico': ['IsSevereCopd','IsAsthma','Is History Of Pneumonia'],
    'comorb_metabolico_endocrino': ['IsDiabetesUncomplicated','IsDiabetesComplicated',
        'IsMorbidObesity','Is Hypothyroidism','Is Hyperthyroidism','Is Dyslipidemias'],
    'comorb_imunossupressao': ['IsImmunossupression','IsSteroidsUse','IsAids'],
    'comorb_transplante': ['Is Solid Organ Transplant','Is Autologous BMT','Is Allogeneic BMT',
        'Is Other Solid Organ Transplant','Is Cardiac Transplant','Is Combined Liverkidney Transplant',
        'Is Combined Pancreaskidney Transplant','Is Liver Transplant','Is Intestinal Transplant',
        'Is Pancreas Transplant','Is Lung Transplant','Is Kidney Transplant'],
    'comorb_neurologico': ['IsStrokeSequelae','IsStrokeNoSequelae','IsDementia'],
    'comorb_psiquiatrico_substancias': ['IsTobaccoConsumption','IsPsychoativeSubstanceDependence',
        'IsAlcoholism','IsPsychiatricDisease'],
    'comorb_outras': ['IsRheumaticDisease','Is Peptic Disease','IsMalnourishment'],
}
TODOS_FLAGS_COMORB = [c for grupo in COMORB_GRUPOS.values() for c in grupo]
faltando = [c for c in TODOS_FLAGS_COMORB if c not in base_integrada.columns]
assert not faltando, f'Comorbidity columns not found: {faltando}'

base_integrada[TODOS_FLAGS_COMORB] = base_integrada[TODOS_FLAGS_COMORB].fillna(0)
base_integrada['n_comorbidades'] = base_integrada[TODOS_FLAGS_COMORB].sum(axis=1)
for grupo, flags in COMORB_GRUPOS.items():
    base_integrada[grupo] = (base_integrada[flags].sum(axis=1) > 0).astype(int)

base_integrada['shock_index'] = (base_integrada['Highest Heart Rate1h'] /
                                  base_integrada['Lowest Systolic Blood Pressure1h'])
base_integrada.loc[base_integrada['Lowest Systolic Blood Pressure1h'] <= 0, 'shock_index'] = np.nan

base_integrada['razao_ureia_creatinina'] = base_integrada['Urea'] / base_integrada['Highest Creatinine1h']
base_integrada.loc[base_integrada['Highest Creatinine1h'] <= 0, 'razao_ureia_creatinina'] = np.nan

print('Comorbidities and physiological ratios derived.')

In [ ]:
GRANDE_AREA_MAP = {
    'Cardiovascular':'cardiovascular', 'Cardiorespiratory arrest':'cardiovascular',
    'Choque (exceto sepse)':'cardiovascular', 'Cardiac surgeries (except congenital heart disease)':'cardiovascular',
    'Combined cardiac surgeries':'cardiovascular', 'Cardiac surgeries for congenital heart disease':'cardiovascular',
    'Invasive cardiac and endovascular procedures':'cardiovascular', 'Cirurgias vasculares':'cardiovascular',
    'Procedimentos / cirurgias endovasculares':'cardiovascular', 'Congenital heart disease':'cardiovascular',
    'Respiratory (except sepsis / infection)':'respiratorio', 'Lung, tracheal, and bronchial surgeries':'respiratorio',
    'Other thoracic surgeries':'respiratorio',
    'Infection / sepsis':'infeccioso',
    'Neurological':'neurologico', 'Neurocirurgias':'neurologico', 'Brain death':'neurologico',
    'Renal':'renal_geniturinario', 'Male genital organ diseases':'renal_geniturinario',
    'Female genital organ diseases':'renal_geniturinario', 'Urinary tract surgeries':'renal_geniturinario',
    'Prostate surgeries':'renal_geniturinario', 'Gynecological and breast surgeries':'renal_geniturinario',
    'Cirurgias da mama':'renal_geniturinario', 'Breast diseases':'renal_geniturinario',
    'Obstetrics':'renal_geniturinario', 'Obstetric surgeries and procedures':'renal_geniturinario',
    'Gastrointestinal':'gastro_hepatobiliar', 'Liver, Biliary Tract, Spleen, and Pancreas':'gastro_hepatobiliar',
    'Abdominal / retroperitoneal surgeries':'gastro_hepatobiliar', 'Colon, sigmoid, rectal, and anal canal surgeries':'gastro_hepatobiliar',
    'Abdominal wall hernias and defects':'gastro_hepatobiliar', 'Gastric surgeries':'gastro_hepatobiliar',
    'Cirurgias do intestino delgado':'gastro_hepatobiliar', 'Liver, biliary tract, and pancreas surgeries':'gastro_hepatobiliar',
    'Repair of abdominal wall hernias and defects':'gastro_hepatobiliar', 'Bariatric surgeries':'gastro_hepatobiliar',
    'Esophageal surgeries':'gastro_hepatobiliar',
    'Hematological':'onco_hematologico', 'Tumors and oncological complications':'onco_hematologico',
    'Endocrine / metabolic':'metabolico_endocrino', 'Endocrine gland surgeries':'metabolico_endocrino',
    'Psychiatric diseases':'psiquiatrico',
    'Causas externas':'trauma_externas', 'Non-surgical trauma':'trauma_externas', 'Surgical polytrauma':'trauma_externas',
    'Outros / diversos':'outros', 'Procedimentos invasivos (exceto cardiovascular)':'outros',
    'Cuidados paliativos':'outros', 'Ophthalmological':'outros', 'Outras Cirurgias':'outros',
    'Autoimmune diseases and collagenoses':'outros', 'Multiple organ dysfunction':'outros',
    'Ear, nose, and mouth diseases':'outros', 'Orthopedic surgeries':'outros',
    'Osteoarticular and soft tissue diseases':'outros', 'Cirurgias da pele e tecidos moles':'outros',
    'Head and neck surgeries':'outros', 'Allergic and hypersensitivity reactions':'outros',
    'Skin diseases (except allergic and hypersensitivity reactions)':'outros', 'Monitoring':'outros',
    'Cirurgias de coluna':'outros', 'Otorhinolaryngological surgeries':'outros',
}

def is_true(s):
    return s.fillna(0).astype(float).eq(1)

folha8 = raw['diag_secund']
registros = []
for i in range(1, N_POSICOES+1):
    rcol, pcol = f'Admission Reason Name{i}', f'IsSecondaryDiagAtICU|Admission{i}'
    if rcol not in folha8.columns or pcol not in folha8.columns:
        continue
    presente = is_true(folha8[pcol])
    grupo_i = folha8[rcol].map(GRANDE_AREA_MAP)
    registros.append(pd.DataFrame({'_linha_bruta': folha8['_linha_bruta'], 'presente': presente, 'grupo': grupo_i}))
long_df = pd.concat(registros, ignore_index=True)
long_df = long_df[long_df['presente']]

contagem = long_df.groupby('_linha_bruta').size().rename('n_diag_secund_admissao')
grupos_wide = (long_df.dropna(subset=['grupo']).assign(valor=1)
               .pivot_table(index='_linha_bruta', columns='grupo', values='valor', aggfunc='max', fill_value=0))
grupos_wide.columns = [f'diag2_{g}' for g in grupos_wide.columns]
diag_secund_por_linha = contagem.to_frame().join(grupos_wide, how='outer').reset_index()
diag_cols = [c for c in diag_secund_por_linha.columns if c != '_linha_bruta']

folha8_chave_por_linha = folha8.set_index('_linha_bruta')[['EpimedCode','MedicalRecord']]
alvo_chk = folha8_chave_por_linha.reindex(base_integrada['_linha_bruta']).reset_index(drop=True)
bate = ((alvo_chk['EpimedCode'].values == base_integrada['EpimedCode'].values) &
        (alvo_chk['MedicalRecord'].values == base_integrada['MedicalRecord'].values))
base_integrada['folha8_integravel'] = bate

base_integrada = base_integrada.merge(diag_secund_por_linha, on='_linha_bruta', how='left', validate='one_to_one')
base_integrada.loc[~base_integrada['folha8_integravel'], diag_cols] = 0
base_integrada[diag_cols] = base_integrada[diag_cols].fillna(0)

VARS_INDICADOR = [
    'BMI','Height','Weight','Lowest Systolic Blood Pressure1h','Lowest Diastolic Blood Pressure1h',
    'Lowest Mean Arterial Pressure1h','Highest Heart Rate1h','Highest Respiratory Rate1h',
    'Highest Temperature1h','Lowest Glasgow Coma Scale1h','Highest Leukocyte Count1h',
    'Lowest Platelets Count1h','Highest Creatinine1h','Highest Bilirubin1h','Highest PH1h',
    'Highest Pa O21h','Highest Pa CO21h','Highest Fi O21h','Highest Pa O2Fi O21h',
    'Highest Arterial Lactate1h','Urea','BUN','Charlson Comorbidity Index','Sofa Score',
    'shock_index','razao_ureia_creatinina',
]
VARS_INDICADOR = [c for c in VARS_INDICADOR if c in base_integrada.columns]
for col in VARS_INDICADOR:
    if base_integrada[col].isna().any():
        base_integrada[f'{col}_ausente'] = base_integrada[col].isna().astype(int)

if 'Chronic Health Status Name' in base_integrada.columns:
    base_integrada['Chronic Health Status Name'] = (
        base_integrada['Chronic Health Status Name'].astype('string').fillna('Not reported'))

if 'ano' not in base_integrada.columns:
    base_integrada['ano'] = pd.to_datetime(base_integrada['Month Year'], format='%m/%Y').dt.year

print('Feature engineering (Phase 2) complete.')

## 5 · Mortality by admission reason — original vs. adjusted

Before any model: the direct effect of the redefinition on observed mortality per category, the finding
that motivated this analysis.

In [ ]:
def intervalo_wilson(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = k / n
    denom = 1 + z**2 / n
    centro = (p + z**2 / (2 * n)) / denom
    margem = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denom
    return max(0, centro - margem), min(1, centro + margem)


def mortalidade_por_motivo(df_in, col_desfecho, n_min=20):
    linhas = []
    for categoria, sub in df_in.groupby('AdmissionReasonName'):
        sub_conhecido = sub[sub[col_desfecho].notna()]
        n = len(sub_conhecido)
        if n < n_min:
            continue
        k = int(sub_conhecido[col_desfecho].sum())
        ic_low, ic_high = intervalo_wilson(k, n)
        linhas.append({'categoria': categoria, 'n': n, 'taxa_obito': k / n, 'ic_low': ic_low, 'ic_high': ic_high})
    return pd.DataFrame(linhas).sort_values('taxa_obito', ascending=False).reset_index(drop=True)


tab_original = mortalidade_por_motivo(base_integrada, 'mortalidade_hospitalar_original')
tab_ajustada = mortalidade_por_motivo(base_integrada, 'mortalidade_hospitalar_ajustada')

comparacao_motivo = tab_original.merge(
    tab_ajustada, on='categoria', suffixes=('_original', '_ajustada'), how='outer'
).sort_values('taxa_obito_ajustada', ascending=False)

print(comparacao_motivo.to_string(index=False))
comparacao_motivo_path = f'{OUTDIR_PARTE2}/mortalidade_por_motivo_ajustada.csv'
comparacao_motivo.to_csv(comparacao_motivo_path, index=False, encoding='utf-8-sig')
print(f'\\nTabela salva: {comparacao_motivo_path}')

In [ ]:
# --- Plot: mortality by admission reason, original vs. adjusted, side by side ---
top_motivos = comparacao_motivo.dropna(subset=['taxa_obito_ajustada']).head(12)
y_pos = np.arange(len(top_motivos))

fig, ax = plt.subplots(figsize=(9, 7))
ax.errorbar(top_motivos['taxa_obito_original'] * 100, y_pos - 0.12,
            xerr=np.vstack([top_motivos['taxa_obito_original'] - top_motivos['ic_low_original'],
                             top_motivos['ic_high_original'] - top_motivos['taxa_obito_original']]) * 100,
            fmt='o', color='#8FAADC', ecolor='#8FAADC', capsize=3, label='Original (transferidos = sobreviventes)')
ax.errorbar(top_motivos['taxa_obito_ajustada'] * 100, y_pos + 0.12,
            xerr=np.vstack([top_motivos['taxa_obito_ajustada'] - top_motivos['ic_low_ajustada'],
                             top_motivos['ic_high_ajustada'] - top_motivos['taxa_obito_ajustada']]) * 100,
            fmt='s', color='#C00000', ecolor='#C00000', capsize=3, label='Adjusted (transferred excluded)')

ax.set_yticks(y_pos)
ax.set_yticklabels(top_motivos['categoria'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Mortalidade hospitalar (%)')
ax.set_title('Mortality by admission reason — original vs. adjusted\n(transferred to another hospital)')
ax.legend(fontsize=9, loc='lower right')
plt.tight_layout()

fig_motivo_path = f'{OUTDIR_PARTE2}/mortalidade_por_motivo_ajustada.png'
fig.savefig(fig_motivo_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura salva: {fig_motivo_path}')

## 6 · Refitting the winning engines on the adjusted cohort

Same hyperparameters already winning in Phase 4 (no new search) — the goal is to test the robustness of
the already-chosen configuration against the outcome redefinition, not to optimize a new model.

In [ ]:
import joblib
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, brier_score_loss

with open(CONJ_PATH, encoding='utf-8') as f:
    sets = json.load(f)
CONJUNTO_1 = sets['set_1_escores']
CONJUNTO_2 = sets['set_2_bruto']
BENCHMARK  = sets['benchmark']

def usar_diagnostico_agrupado(set):
    return ['AdmissionMainDiagnosisName_grp' if c == 'AdmissionMainDiagnosisName' else c for c in set]

CONJUNTO_1 = usar_diagnostico_agrupado(CONJUNTO_1)
CONJUNTO_2 = usar_diagnostico_agrupado(CONJUNTO_2)
CONJUNTOS_VAR = {'Conjunto 1 (escores)': CONJUNTO_1, 'Conjunto 2 (brutos)': CONJUNTO_2}

with open(f'{FASE4_DIR}/modelo_A_fase4_melhores_motores.json', encoding='utf-8') as f:
    melhor_por_set = json.load(f)

def preparar_boosting(set, treino, teste):
    cats = [c for c in set if c in CATEGORICAS]
    nums = [c for c in set if c not in CATEGORICAS and c != 'Month Year']
    cols = cats + nums
    Xtr = treino[cols].copy()
    Xte = teste[cols].copy()
    for c in cats:
        Xtr[c] = Xtr[c].astype('object').fillna('Not reported')
        Xte[c] = Xte[c].astype('object').fillna('Not reported')
        categorias = pd.Index(Xtr[c].unique()).union(['Not reported'])
        Xte[c] = Xte[c].where(Xte[c].isin(categorias), 'Not reported')
        Xtr[c] = pd.Categorical(Xtr[c], categories=categorias)
        Xte[c] = pd.Categorical(Xte[c], categories=categorias)
    return Xtr, Xte, cats, nums

def agrupar_diagnostico_principal(treino, teste, col='AdmissionMainDiagnosisName', n_top=N_TOP_DIAG):
    col_grp = f'{col}_grp'
    top_categorias = treino[col].value_counts().nlargest(n_top).index
    treino = treino.copy(); teste = teste.copy()
    treino[col_grp] = treino[col].where(treino[col].isin(top_categorias), 'outros')
    teste[col_grp]  = teste[col].where(teste[col].isin(top_categorias), 'outros')
    return treino, teste

print('Helper functions (identical to Phase 4/5) defined.')

In [ ]:
# --- DeLong test functions (identical to Phase 4) ---
def _compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2

def _fast_delong(preds_sorted_transposed, m):
    k = preds_sorted_transposed.shape[0]
    n = preds_sorted_transposed.shape[1] - m
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r, :] = _compute_midrank(preds_sorted_transposed[r, :m])
        ty[r, :] = _compute_midrank(preds_sorted_transposed[r, m:])
        tz[r, :] = _compute_midrank(preds_sorted_transposed[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    sx = np.cov(v01); sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov

def teste_delong(y_true, p1, p2):
    from scipy import stats as sstats
    order = np.argsort(-y_true)
    y_ord = y_true[order]
    m = int(y_ord.sum())
    preds = np.vstack([p1[order], p2[order]])
    aucs, cov = _fast_delong(preds, m)
    diff = aucs[0] - aucs[1]
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    z = diff / np.sqrt(var) if var > 0 else np.nan
    p_valor = 2 * (1 - sstats.norm.cdf(abs(z))) if var > 0 else np.nan
    return aucs[0], aucs[1], diff, z, p_valor

print('DeLong functions defined.')

In [ ]:
import shap
resultados_ajustada = []
previsoes_roc = {}  # stores y/predictions per (definition, set), for the later ROC-curve plot
modelos_parte2, dados_prep_parte2 = {}, {}  # stores model/Xtr/ytr/groups per (definition, set), for recalibration (Section 8)

for nome_col_desfecho, rotulo in [('mortalidade_hospitalar_original', 'Original'),
                                    ('mortalidade_hospitalar_ajustada', 'Adjusted (transferred excluded)')]:
    base_conhecida = base_integrada[base_integrada[nome_col_desfecho].notna()].copy()
    base_conhecida[ALVO] = base_conhecida[nome_col_desfecho].astype(int)

    treino_t = base_conhecida[base_conhecida['ano'] <  CORTE_TEMPORAL].copy()
    teste_t  = base_conhecida[base_conhecida['ano'] >= CORTE_TEMPORAL].copy()
    treino_t, teste_t = agrupar_diagnostico_principal(treino_t, teste_t)

    for nome_set, nome_motor in melhor_por_set.items():
        Xtr, Xte, cats, nums = preparar_boosting(CONJUNTOS_VAR[nome_set], treino_t, teste_t)
        ytr, yte = treino_t[ALVO].values, teste_t[ALVO].values

        modelo_base = joblib.load(f"{FASE4_DIR}/modelo_{nome_set.split()[1].strip('()')}_{nome_motor}.joblib")
        modelo = clone(modelo_base)
        if 'LGBM' in type(modelo).__name__:
            modelo.fit(Xtr, ytr, categorical_feature=cats)
        else:
            modelo.fit(Xtr, ytr)

        p_teste = modelo.predict_proba(Xte)[:, 1]
        p_saps3 = teste_t[BENCHMARK].values / 100

        # --- Store this combination's (definition, set) model and training data, for Section 8 ---
        modelos_parte2[(rotulo, nome_set)] = modelo
        dados_prep_parte2[(rotulo, nome_set)] = {
            'Xtr': Xtr, 'ytr': ytr, 'cats': cats, 'groups_tr': treino_t['patient_id'].values,
            'Xte': Xte, 'yte': yte,  # added for Section 10 (ablation/parsimony), which needs the test set
        }

        # --- Store the predictions for the ROC-curve plot (Original vs. Adjusted) ---
        previsoes_roc[(rotulo, nome_set)] = {'y': yte, 'p_modelo': p_teste, 'p_saps3': p_saps3}

        auroc_modelo, auroc_saps3, diff, z, p_valor = teste_delong(yte, p_teste, p_saps3)

        resultados_ajustada.append({
            'definicao_desfecho': rotulo, 'set': nome_set, 'motor': nome_motor,
            'n_treino': len(treino_t), 'n_teste': len(teste_t), 'taxa_obito_teste': yte.mean(),
            'AUROC_modelo': auroc_modelo, 'AUROC_SAPS3': auroc_saps3,
            'diferenca_vs_SAPS3': diff, 'p_valor_delong': p_valor,
            'Brier': brier_score_loss(yte, p_teste),
        })
        print(f'[{rotulo}] {nome_set} ({nome_motor}): AUROC={auroc_modelo:.4f}  '
              f'vs. SAPS3={auroc_saps3:.4f}  diff={diff:+.4f}  p={p_valor:.4f}  n_teste={len(teste_t)}')

tab_ajustada_modelo = pd.DataFrame(resultados_ajustada)
tab_ajustada_path = f'{OUTDIR_PARTE2}/modelo_ajustado_transferencia.csv'
tab_ajustada_modelo.to_csv(tab_ajustada_path, index=False, encoding='utf-8-sig')
print(f'\nTabela salva: {tab_ajustada_path}')

In [ ]:
# --- Save the full model (Set 2, adjusted) and the associated training/test data,
#     for reuse in Phase 6 ---
ROTULO_SALVAR = 'Adjusted (transferred excluded)'
CONJUNTO_SALVAR = 'Conjunto 2 (brutos)'

dados_salvar = dados_prep_parte2[(ROTULO_SALVAR, CONJUNTO_SALVAR)]
modelo_salvar = modelos_parte2[(ROTULO_SALVAR, CONJUNTO_SALVAR)]
previsoes_salvar = previsoes_roc[(ROTULO_SALVAR, CONJUNTO_SALVAR)]

# teste_t, at this point in the notebook, still corresponds to the last combination processed in Cell 56's loop
# (Adjusted x Set 2) — used here only to retrieve each adjusted-test patient's patient_id,
# which Phase 6 needs to identify which "unadjusted" test patients were excluded by transfer.
assert len(teste_t) == len(dados_salvar['yte']), (
    "teste_t does not correspond to the Adjusted x Set 2 combination — check Cell 56's loop order")

artefato_fase6 = {
    'modelo': modelo_salvar,
    'Xtr': dados_salvar['Xtr'],
    'Xte': dados_salvar['Xte'],
    'ytr': dados_salvar['ytr'],
    'yte': dados_salvar['yte'],
    'cats': dados_salvar['cats'],
    'groups_tr': dados_salvar['groups_tr'],
    'p_teste': previsoes_salvar['p_modelo'],
    'patient_ids_teste': teste_t['patient_id'].values,
}

path_artefato_fase6 = f'{OUTDIR_PARTE2}/modelo_completo_com_ajuste_set2.joblib'
joblib.dump(artefato_fase6, path_artefato_fase6)

print(f'Artifact saved for Phase 6: {path_artefato_fase6}')
print(f"  Modelo: {type(modelo_salvar).__name__}  |  n_treino={len(dados_salvar['ytr'])}  "
      f"n_teste={len(dados_salvar['yte'])}  |  "
      f"test AUROC={roc_auc_score(dados_salvar['yte'], previsoes_salvar['p_modelo']):.4f}")

In [ ]:
# --- Plot: ROC curves — Original vs. Adjusted, per set, model and SAPS3, with DeLong p-value ---
fig, axes = plt.subplots(1, len(CONJUNTOS_VAR), figsize=(7 * len(CONJUNTOS_VAR), 6.5))
if len(CONJUNTOS_VAR) == 1:
    axes = [axes]

cores_definicao = {
    'Original': '#8FB8D9',
    'Adjusted (transferred excluded)': '#C00000',
}

linha_saps3 = None  # single handle for the shared legend — same score, same sample in both panels

for eixo, nome_set in zip(axes, CONJUNTOS_VAR.keys()):
    for rotulo, cor in cores_definicao.items():
        dados = previsoes_roc[(rotulo, nome_set)]
        y, p_modelo, p_saps3 = dados['y'], dados['p_modelo'], dados['p_saps3']

        fpr_m, tpr_m, _ = roc_curve(y, p_modelo)
        auroc_m = roc_auc_score(y, p_modelo)

        linha = tab_ajustada_modelo[(tab_ajustada_modelo['definicao_desfecho'] == rotulo) &
                                     (tab_ajustada_modelo['set'] == nome_set)].iloc[0]
        p_valor_vs_saps3 = linha['p_valor_delong']
        sig = 'p < 0,001' if p_valor_vs_saps3 < 0.001 else f'p = {p_valor_vs_saps3:.3f}'

        eixo.plot(fpr_m, tpr_m, color=cor, linewidth=2.2,
                  label=f'{rotulo}: AUROC={auroc_m:.3f}  (vs. SAPS3 {sig})')

    # SAPS3 (adjusted definition as the single reference): plotted in both panels for visual reading,
    # but NOT entering each panel's legend — it is the same score, same sample, see single legend below.
    dados_ref = previsoes_roc[('Adjusted (transferred excluded)', nome_set)]
    fpr_s, tpr_s, _ = roc_curve(dados_ref['y'], dados_ref['p_saps3'])
    auroc_s = roc_auc_score(dados_ref['y'], dados_ref['p_saps3'])
    linha_atual, = eixo.plot(fpr_s, tpr_s, color='#666666', linestyle='--', linewidth=1.5)
    if linha_saps3 is None:
        linha_saps3 = linha_atual
        auroc_saps3_global = auroc_s

    eixo.plot([0, 1], [0, 1], color='#CCCCCC', linestyle=':', linewidth=1)
    eixo.set_xlabel('1 − Especificidade (FPR)')
    eixo.set_ylabel('Sensibilidade (TPR)')
    eixo.set_title(nome_set, fontsize=11, fontweight='bold')
    eixo.legend(fontsize=8, loc='lower right')  # model curves only; SAPS3 stays in the single legend
    for spine in ['top', 'right']:
        eixo.spines[spine].set_visible(False)

plt.suptitle('ROC curves — original vs. adjusted outcome definition', fontsize=13, fontweight='bold', y=1.04)

# Single SAPS3 legend, shared by both panels, noting it is the same score/sample
fig.legend([linha_saps3], [f'SAPS3: AUROC={auroc_saps3_global:.3f}  '
                           f'— same score, same sample, for both sets'],
           loc='upper center', bbox_to_anchor=(0.5, 0.97), ncol=1, fontsize=8.5, frameon=False)

plt.tight_layout(rect=[0, 0, 1, 0.93])

fig_roc_path = f'{OUTDIR_PARTE2}/fase_vies_transferencia_roc.png'
fig.savefig(fig_roc_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura salva: {fig_roc_path}')

## 7 · Classification metrics at operational thresholds — original vs. adjusted

The same three thresholds as Part I (Youden, ≈0.10, ≈0.20), applied to the predictions already obtained
in Section 6, for both outcome definitions and both variable sets. Reuses `limiar_youden` (defined in
Part I) on the predictions stored in `previsoes_roc`.

In [ ]:
# --- Classification metrics at operational thresholds — original vs. adjusted ---
LIMIARES_CLINICOS_P2 = [0.10, 0.20]

linhas_classificacao_p2 = []
for rotulo in cores_definicao.keys():
    for nome_set in CONJUNTOS_VAR:
        dados = previsoes_roc[(rotulo, nome_set)]
        y, p = dados['y'], dados['p_modelo']

        limiar_y = limiar_youden(y, p)
        limiares_a_testar = [('Youden', limiar_y)] + [(f'{l:.2f}', l) for l in LIMIARES_CLINICOS_P2]

        for nome_limiar, limiar in limiares_a_testar:
            pred = (p >= limiar).astype(int)
            tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
            sens = tp / (tp + fn) if (tp + fn) else np.nan
            espec = tn / (tn + fp) if (tn + fp) else np.nan
            vpp = tp / (tp + fp) if (tp + fp) else np.nan
            vpn = tn / (tn + fn) if (tn + fn) else np.nan
            linhas_classificacao_p2.append({
                'definicao_desfecho': rotulo, 'set': nome_set,
                'limiar_nome': nome_limiar, 'limiar_valor': limiar,
                'VP': tp, 'FP': fp, 'VN': tn, 'FN': fn,
                'sensibilidade': sens, 'especificidade': espec, 'VPP': vpp, 'VPN': vpn,
                'accuracy': accuracy_score(y, pred), 'F1': f1_score(y, pred),
                'MCC': matthews_corrcoef(y, pred),
            })

tab_classificacao_p2 = pd.DataFrame(linhas_classificacao_p2)
classif_p2_path = f'{OUTDIR_PARTE2}/modelo_A_fase5_classificacao_ajustada.csv'
tab_classificacao_p2.to_csv(classif_p2_path, index=False, encoding='utf-8-sig')
print(f'Classification table (original vs. adjusted) saved: {classif_p2_path}')
print(tab_classificacao_p2.to_string(index=False))
print()
print('Note: same caveat as Part I — accuracy and F1 uninformative at this prevalence; '
      'MCC is the preferred scalar summary.')


### Note — reading the classification table (original vs. adjusted)

**Internal consistency.** The six rows labeled `Original` in this table are identical, cell by cell (same
thresholds, same TP/FP/TN/FN counts), to those in Part I's classification table. This confirms that
Section 6's refit correctly reproduces Phase 4's frozen model.

**The Youden thresholds are not the same across the two definitions.** Each threshold is recomputed on
its respective test set (Youden maximizes sensitivity + specificity *in that* sample). Since the sample
changes — 1,400 patients in the original definition, 1,277 in the adjusted one (the 123 transferred
patients of the test period leave) — the cut point also changes: for example, Set 1 goes from 0.139
(Original) to 0.185 (Adjusted). The fixed thresholds (0.10 and 0.20) remain comparable across the two
definitions; Youden's does not.

**The absolute counts (TP/FP/TN/FN) are not directly comparable between Original and Adjusted.** The
denominator changes (n=1,400 vs. n=1,277), and the drop concentrates mostly in true negatives (Set 1:
930→888; Set 2: 1,060→932) — expected, since the excluded transferred patients were, under the original
definition, counted as survivors (potential TNs or FNs). To compare the two definitions, use the rates
(sensitivity, specificity, PPV, NPV, MCC), not the raw counts.

**The effect of the adjusted definition is not uniform across the two variable sets**, at the Youden
threshold:
- Set 1 (scores): MCC improves from 0.383 to 0.418 — specificity rises more (0.757→0.803) than
  sensitivity falls (0.784→0.743).
- Set 2 (raw): MCC worsens slightly, from 0.546 to 0.540 — here sensitivity rises (0.830→0.842), but
  specificity falls a bit (0.862→0.843), and the net effect on MCC is discretely negative.

In short: excluding transferred patients does not make classification uniformly "better" — it shifts the
sensitivity/specificity balance differently depending on the variable set, and the effect on the scalar
summary (MCC) is small and of opposite sign between the two sets. Since this table reports only point
estimates, without confidence intervals, these MCC differences (0.035 and −0.006) should be read with
caution — the recalibration section and the DeLong test (Section 8) are the most robust references for
stating whether the adjusted definition truly changes model performance, or whether these fluctuations
are within the expected sampling noise between 1,277 and 1,400 test patients.

## 8 · Post-hoc recalibration — original vs. adjusted

Same decision criterion as Part I (95% bootstrap CI of the calibration intercept/slope) and the same
correction procedure (Platt scaling on *out-of-fold* training predictions, never on the test set),
applied here to the four combinations (Original × Adjusted) × (Set 1 × Set 2). Reuses
`inclinacao_intercepto_calibracao` and `obter_predicoes_out_of_fold`, already defined in Part I.

In [ ]:
# --- Recalibration decision (95% bootstrap CI) — original vs. adjusted ---
linhas_decisao_recal_p2 = []
for rotulo in cores_definicao.keys():
    for nome_set in CONJUNTOS_VAR:
        dados = previsoes_roc[(rotulo, nome_set)]
        y, p = dados['y'], dados['p_modelo']
        intercepto, inclinacao = inclinacao_intercepto_calibracao(y, p)

        rng = np.random.default_rng(SEED)
        n = len(y)
        interceptos_boot, inclinacoes_boot = [], []
        for _ in range(N_BOOT):
            idx = rng.integers(0, n, n)
            if len(np.unique(y[idx])) < 2:
                continue
            i_b, s_b = inclinacao_intercepto_calibracao(y[idx], p[idx])
            interceptos_boot.append(i_b); inclinacoes_boot.append(s_b)

        ic_intercepto = np.percentile(interceptos_boot, [2.5, 97.5])
        ic_inclinacao = np.percentile(inclinacoes_boot, [2.5, 97.5])
        recalibrar = (ic_intercepto[0] > 0 or ic_intercepto[1] < 0) or (ic_inclinacao[0] > 1 or ic_inclinacao[1] < 1)

        linhas_decisao_recal_p2.append({
            'definicao_desfecho': rotulo, 'set': nome_set,
            'intercepto': intercepto, 'inclinacao': inclinacao,
            'IC_intercepto_low': ic_intercepto[0], 'IC_intercepto_high': ic_intercepto[1],
            'IC_inclinacao_low': ic_inclinacao[0], 'IC_inclinacao_high': ic_inclinacao[1],
            'recalibrar': recalibrar,
        })
        print(f"[{rotulo}] {nome_set}: intercepto={intercepto:.3f} IC95%[{ic_intercepto[0]:.3f}, {ic_intercepto[1]:.3f}] | "
              f"slope={inclinacao:.3f} 95%CI[{ic_inclinacao[0]:.3f}, {ic_inclinacao[1]:.3f}] | recalibrate={recalibrar}")

tab_decisao_recal_p2 = pd.DataFrame(linhas_decisao_recal_p2)


In [ ]:
# --- Platt scaling correction (out-of-fold), only where the decision above indicated recalibration ---
linhas_recalibracao_p2 = []
predicoes_recalibradas_p2 = {}

for rotulo in cores_definicao.keys():
    for nome_set in CONJUNTOS_VAR:
        chave = (rotulo, nome_set)
        linha_decisao = tab_decisao_recal_p2[
            (tab_decisao_recal_p2['definicao_desfecho'] == rotulo) &
            (tab_decisao_recal_p2['set'] == nome_set)].iloc[0]
        dados = previsoes_roc[chave]
        y, p = dados['y'], dados['p_modelo']

        if not linha_decisao['recalibrar']:
            print(f'[{rotulo}] {nome_set}: 95% CI compatible with adequate calibration — recalibration not applied.')
            predicoes_recalibradas_p2[chave] = p
            continue

        prep = dados_prep_parte2[chave]
        modelo = modelos_parte2[chave]
        p_oof = obter_predicoes_out_of_fold(modelo, prep['Xtr'], prep['ytr'], prep['groups_tr'], prep['cats'])
        intercepto_platt, inclinacao_platt = inclinacao_intercepto_calibracao(prep['ytr'], p_oof)

        logit_p = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
        logit_recal = intercepto_platt + inclinacao_platt * logit_p
        p_recal = 1 / (1 + np.exp(-logit_recal))
        predicoes_recalibradas_p2[chave] = p_recal

        brier_antes = brier_score_loss(y, p)
        brier_depois = brier_score_loss(y, p_recal)
        linhas_recalibracao_p2.append({
            'definicao_desfecho': rotulo, 'set': nome_set,
            'intercepto_platt_oof': intercepto_platt, 'inclinacao_platt_oof': inclinacao_platt,
            'Brier_antes': brier_antes, 'Brier_depois': brier_depois,
        })
        print(f'[{rotulo}] {nome_set}: recalibrado (Platt out-of-fold). Brier {brier_antes:.4f} -> {brier_depois:.4f}')

tab_recalibracao_p2 = pd.DataFrame(linhas_recalibracao_p2)
recal_p2_path = f'{OUTDIR_PARTE2}/modelo_A_fase5_recalibracao_ajustada.csv'
decisao_p2_path = f'{OUTDIR_PARTE2}/modelo_A_fase5_decisao_recalibracao_ajustada.csv'
tab_recalibracao_p2.to_csv(recal_p2_path, index=False, encoding='utf-8-sig')
tab_decisao_recal_p2.to_csv(decisao_p2_path, index=False, encoding='utf-8-sig')
print(f'Tabelas salvas: {recal_p2_path} | {decisao_p2_path}')


## 9 · Top mortality categories — original vs. adjusted

Stylized version of Section 5's comparison (same data as `comparacao_motivo`), restricted to the
categories with highest mortality under the adjusted definition — in the same visual pattern as Part I's
`top10` panel.

In [ ]:
# --- Top N categories by mortality (adjusted definition) — original vs. adjusted, side by side ---
from matplotlib.transforms import blended_transform_factory

N_TOP_MOTIVO = 10

top_mortalidade_ajustada = (comparacao_motivo.dropna(subset=['taxa_obito_ajustada'])
                             .sort_values('taxa_obito_ajustada', ascending=False)
                             .head(N_TOP_MOTIVO).iloc[::-1].reset_index(drop=True))

y_pos = np.arange(len(top_mortalidade_ajustada))
fig, ax = plt.subplots(figsize=(9.5, 7))

ax.errorbar(top_mortalidade_ajustada['taxa_obito_original'] * 100, y_pos - 0.12,
            xerr=np.vstack([top_mortalidade_ajustada['taxa_obito_original'] - top_mortalidade_ajustada['ic_low_original'],
                            top_mortalidade_ajustada['ic_high_original'] - top_mortalidade_ajustada['taxa_obito_original']]) * 100,
            fmt='o', color='#8FAADC', ecolor='#8FAADC', capsize=3, markersize=6,
            label='Original (transferidos = sobreviventes)')
ax.errorbar(top_mortalidade_ajustada['taxa_obito_ajustada'] * 100, y_pos + 0.12,
            xerr=np.vstack([top_mortalidade_ajustada['taxa_obito_ajustada'] - top_mortalidade_ajustada['ic_low_ajustada'],
                            top_mortalidade_ajustada['ic_high_ajustada'] - top_mortalidade_ajustada['taxa_obito_ajustada']]) * 100,
            fmt='s', color='#C00000', ecolor='#C00000', capsize=3, markersize=6,
            label='Adjusted (transferred excluded)')

# --- n labels: outside the plotting area, in axis coordinates (not data coordinates) ---
# Before, the text sat at x_max*0.99 (data coordinates), so long error bars
# (e.g., cardiac arrest) invaded the text's space. Placing the text in
# axis coordinates (blended transform: x fixed as an axis fraction, y still in data), it
# always sits to the right of any error bar, in a band reserved for the labels alone.
ax.set_xlim(left=0)  # fixes the limit before computing the reserved band
x_min, x_max_dado = ax.get_xlim()
ax.set_xlim(x_min, x_max_dado * 1.18)  # opens ~18% of space on the right, for the n labels alone

trans = blended_transform_factory(ax.transAxes, ax.transData)
for y, (n_o, n_a) in enumerate(zip(top_mortalidade_ajustada['n_original'], top_mortalidade_ajustada['n_ajustada'])):
    n_a_fmt = f'{int(n_a)}' if pd.notna(n_a) else '—'
    ax.text(1.01, y, f'n={int(n_o)}\u2192{n_a_fmt}', va='center', fontsize=7.5, color='#555555',
            ha='left', transform=trans, clip_on=False)

ax.set_yticks(y_pos)
ax.set_yticklabels(top_mortalidade_ajustada['categoria'], fontsize=9)
ax.set_xlabel('Mortalidade hospitalar (%)')
ax.set_title(f'Top {N_TOP_MOTIVO} admission reasons by mortality (adjusted definition)\n'
             'original vs. adjusted (transferred excluded)', fontsize=11, fontweight='bold')
ax.legend(fontsize=8.5, loc='lower right')
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.grid(axis='x', linestyle=':', alpha=0.4)

plt.tight_layout()
fig_top_motivo_path = f'{OUTDIR_PARTE2}/fase5_top_motivos_mortalidade_ajustada.png'
fig.savefig(fig_top_motivo_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura salva: {fig_top_motivo_path}')

## 10 · Ablation and parsimony — adjusted cohort

Part I showed that a simplified score (N=30 variables, ranked by mean |SHAP|) sustains an AUROC
statistically equivalent to the full model, by DeLong. This section repeats the same sequence for the
adjusted outcome definition — ablation curve, ROC curve of the N=30 score against the full model, and
decision of the smallest non-inferior N — to check whether that parsimony holds when transferred patients
are excluded. At the end, the top-15 SHAP importance composition is compared between the two definitions,
to assess whether the simplified score's most relevant variables change.

### Note — ablation curve and simplified score (adjusted cohort)

Same procedure as Part I, applied to the engine refitted under the adjusted definition (Section 6):
importance ranking by mean |SHAP| on the adjusted training set, refitting on subsets of increasing size,
and comparing each subset's AUROC against the adjusted full model's.

In [ ]:
# --- Ablation curve: AUROC vs. number of variables (ranking by mean |SHAP|) — adjusted cohort ---
from sklearn.base import clone

NOME_CONJUNTO_ABL = 'Conjunto 2 (brutos)'
ROTULO_AJUSTADA = 'Adjusted (transferred excluded)'

dados_ajustada = dados_prep_parte2[(ROTULO_AJUSTADA, NOME_CONJUNTO_ABL)]
Xtr_adj = dados_ajustada['Xtr']
Xte_adj = dados_ajustada['Xte']
cats_adj = dados_ajustada['cats']
ytr_adj = dados_ajustada['ytr']
yte_adj = dados_ajustada['yte']

modelo_base_adj = modelos_parte2[(ROTULO_AJUSTADA, NOME_CONJUNTO_ABL)]

# Full-model reference — predictions already generated in Section 6, no refitting here
n_variaveis_completo_adj = Xtr_adj.shape[1]
p_teste_completo_adj = previsoes_roc[(ROTULO_AJUSTADA, NOME_CONJUNTO_ABL)]['p_modelo']
auroc_completo_adj = roc_auc_score(yte_adj, p_teste_completo_adj)

TAMANHOS_N = [5, 8, 10, 15, 20, 30, 50, 75, 100, n_variaveis_completo_adj]

explainer_adj = shap.TreeExplainer(modelo_base_adj)
shap_vals_treino_adj = explainer_adj(Xtr_adj)
importancia_media_adj = pd.Series(np.abs(shap_vals_treino_adj.values).mean(axis=0), index=Xtr_adj.columns)
ranking_variaveis_ajustada = importancia_media_adj.sort_values(ascending=False).index.tolist()

resultados_ablacao_adj = []
for n in TAMANHOS_N:
    n = min(n, len(ranking_variaveis_ajustada))
    variaveis_n = ranking_variaveis_ajustada[:n]
    cats_n = [c for c in cats_adj if c in variaveis_n]

    Xtr_n = Xtr_adj[variaveis_n]
    Xte_n = Xte_adj[variaveis_n]

    modelo_n = clone(modelo_base_adj)
    if 'LGBM' in type(modelo_n).__name__:
        modelo_n.fit(Xtr_n, ytr_adj, categorical_feature=cats_n)
    else:
        modelo_n.fit(Xtr_n, ytr_adj)

    p_n = modelo_n.predict_proba(Xte_n)[:, 1]
    resultados_ablacao_adj.append({'n_variaveis': n, 'AUROC': roc_auc_score(yte_adj, p_n)})
    print(f'n={n:>3d}  AUROC={resultados_ablacao_adj[-1]["AUROC"]:.4f}')

tab_ablacao_adj = pd.DataFrame(resultados_ablacao_adj).drop_duplicates(subset='n_variaveis').sort_values('n_variaveis')

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot(tab_ablacao_adj['n_variaveis'], tab_ablacao_adj['AUROC'], 'o-', color='#C00000', linewidth=2)
ax.axhline(auroc_completo_adj, color='gray', linestyle='--', linewidth=1,
           label=f'Modelo completo (n={n_variaveis_completo_adj})  AUROC={auroc_completo_adj:.3f}')
ax.set_xlabel('Number of variables (top-N by mean |SHAP|)')
ax.set_ylabel('AUROC (temporal validation)')
ax.set_title(f'Ablation curve — {NOME_CONJUNTO_ABL} (adjusted definition)')
ax.legend(fontsize=9)

plt.tight_layout()
fig_ablacao_adj_path = f'{OUTDIR_PARTE2}/fase6_curva_ablacao_ajustada_{NOME_CONJUNTO_ABL.split()[1].strip("()")}.png'
fig.savefig(fig_ablacao_adj_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura salva: {fig_ablacao_adj_path}')
print(tab_ablacao_adj.to_string(index=False))

In [ ]:
# --- ROC curve — full model vs. simplified score (N=30), adjusted cohort ---
from sklearn.metrics import roc_curve

N_TOP = 30
variaveis_top_adj = ranking_variaveis_ajustada[:N_TOP]
cats_top_adj = [c for c in cats_adj if c in variaveis_top_adj]

Xtr_reduzido_adj = Xtr_adj[variaveis_top_adj]
Xte_reduzido_adj = Xte_adj[variaveis_top_adj]

modelo_reduzido_adj = clone(modelo_base_adj)
if 'LGBM' in type(modelo_reduzido_adj).__name__:
    modelo_reduzido_adj.fit(Xtr_reduzido_adj, ytr_adj, categorical_feature=cats_top_adj)
else:
    modelo_reduzido_adj.fit(Xtr_reduzido_adj, ytr_adj)

p_teste_reduzido_adj = modelo_reduzido_adj.predict_proba(Xte_reduzido_adj)[:, 1]
auroc_reduzido_adj = roc_auc_score(yte_adj, p_teste_reduzido_adj)

print(f'Modelo completo (n={n_variaveis_completo_adj}), ajustada:      AUROC = {auroc_completo_adj:.4f}')
print(f'Escore simplificado (n={N_TOP}), ajustada:            AUROC = {auroc_reduzido_adj:.4f}')
print(f'Simplified score variables (adjusted): {variaveis_top_adj}')

fpr_completo_adj, tpr_completo_adj, _ = roc_curve(yte_adj, p_teste_completo_adj)
fpr_reduzido_adj, tpr_reduzido_adj, _ = roc_curve(yte_adj, p_teste_reduzido_adj)

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot(fpr_completo_adj, tpr_completo_adj, color='#1F3864', linewidth=2.2,
        label=f'Modelo completo (n={n_variaveis_completo_adj})  AUROC={auroc_completo_adj:.3f}')
ax.plot(fpr_reduzido_adj, tpr_reduzido_adj, color='#C00000', linewidth=2.2, linestyle='--',
        label=f'Escore simplificado (n={N_TOP})  AUROC={auroc_reduzido_adj:.3f}')
ax.plot([0, 1], [0, 1], color='gray', linewidth=1, linestyle=':')

ax.set_xlabel('1 − especificidade (FPR)')
ax.set_ylabel('Sensibilidade (TPR)')
ax.set_title('ROC curve — full model vs. simplified score (n=30)\nadjusted definition, temporal validation')
ax.legend(loc='lower right', fontsize=9)

plt.tight_layout()
fig_parsimonia_adj_path = f'{OUTDIR_PARTE2}/fase6_roc_completo_vs_simplificado_ajustada.png'
fig.savefig(fig_parsimonia_adj_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura salva: {fig_parsimonia_adj_path}')

### Note — statistical significance of N=30 in this figure

The plot above is descriptive only: it shows the two ROC curves and each one's AUROC, with no associated
statistical test. The difference between the curves (AUROC 0.907 vs. 0.901) looks small visually, but
that does not replace a paired test.

The p-value corresponding to this same N=30 is in the next cell's candidate table (row
`n_variaveis=30`): **difference=−0.006282, p=0.014** — statistically significant
(`nao_significativo=False`), not a confirmed equivalence. The smallest N passing the 0.05 cut in this
table is N=40 (p=0.250), but the next note already treats that result as isolated sampling noise, not as
a real point of equivalence — N=30 was kept on visual criteria (elbow of the ablation curve) and
comparability grounds, not on direct statistical confirmation in this test.

In [ ]:
# --- N decision of the simplified score: DeLong against the full model (adjusted definition) ---
CANDIDATOS_N = [20, 22, 24, 26, 28, 30]

resultados_candidatos_n_adj = []
for n in CANDIDATOS_N:
    variaveis_n = ranking_variaveis_ajustada[:n]
    cats_n = [c for c in cats_adj if c in variaveis_n]
    Xtr_n, Xte_n = Xtr_adj[variaveis_n], Xte_adj[variaveis_n]

    modelo_n = clone(modelo_base_adj)
    if 'LGBM' in type(modelo_n).__name__:
        modelo_n.fit(Xtr_n, ytr_adj, categorical_feature=cats_n)
    else:
        modelo_n.fit(Xtr_n, ytr_adj)

    p_n = modelo_n.predict_proba(Xte_n)[:, 1]
    # teste_delong (Section 6, defined in '1a966af8') returns (auroc1, auroc2, diff, z, p_value)
    _, _, diff, _, p_valor = teste_delong(yte_adj, p_n, p_teste_completo_adj)
    resultados_candidatos_n_adj.append({
        'n_variaveis': n,
        'AUROC': roc_auc_score(yte_adj, p_n),
        'diferenca_vs_completo': diff,
        'p_valor': p_valor,
        'nao_significativo': p_valor >= 0.05,
    })

tab_candidatos_n_adj = pd.DataFrame(resultados_candidatos_n_adj)
print(tab_candidatos_n_adj.to_string(index=False))

candidatos_ok_adj = tab_candidatos_n_adj[tab_candidatos_n_adj['nao_significativo']]
if len(candidatos_ok_adj):
    n_sugerido_adj = candidatos_ok_adj['n_variaveis'].min()
    print(f'\nSmallest N with no statistically significant difference vs. the full model (adjusted): {n_sugerido_adj}')
else:
    print('\nNenhum candidato testado ficou estatisticamente equivalente ao modelo completo (ajustada) — '
          'considere testar valores maiores.')

tab_candidatos_adj_path = f'{OUTDIR_PARTE2}/fase6_candidatos_n_delong_ajustada.csv'
tab_candidatos_n_adj.to_csv(tab_candidatos_adj_path, index=False, encoding='utf-8-sig')
print(f'Tabela salva: {tab_candidatos_adj_path}')

In [ ]:
# --- N decision of the simplified score: DeLong against the full model (adjusted definition) ---
CANDIDATOS_N = [20, 22, 24, 26, 28, 30, 35, 40, 50, 60]  # expanded — N=30 was still not enough

resultados_candidatos_n_adj = []
for n in CANDIDATOS_N:
    n = min(n, len(ranking_variaveis_ajustada))  # guards against a candidate exceeding the total variable count
    variaveis_n = ranking_variaveis_ajustada[:n]
    cats_n = [c for c in cats_adj if c in variaveis_n]
    Xtr_n, Xte_n = Xtr_adj[variaveis_n], Xte_adj[variaveis_n]

    modelo_n = clone(modelo_base_adj)
    if 'LGBM' in type(modelo_n).__name__:
        modelo_n.fit(Xtr_n, ytr_adj, categorical_feature=cats_n)
    else:
        modelo_n.fit(Xtr_n, ytr_adj)

    p_n = modelo_n.predict_proba(Xte_n)[:, 1]
    _, _, diff, _, p_valor = teste_delong(yte_adj, p_n, p_teste_completo_adj)
    resultados_candidatos_n_adj.append({
        'n_variaveis': n,
        'AUROC': roc_auc_score(yte_adj, p_n),
        'diferenca_vs_completo': diff,
        'p_valor': p_valor,
        'nao_significativo': p_valor >= 0.05,
    })

tab_candidatos_n_adj = pd.DataFrame(resultados_candidatos_n_adj).drop_duplicates(subset='n_variaveis')
print(tab_candidatos_n_adj.to_string(index=False))

candidatos_ok_adj = tab_candidatos_n_adj[tab_candidatos_n_adj['nao_significativo']]
if len(candidatos_ok_adj):
    n_sugerido_adj = candidatos_ok_adj['n_variaveis'].min()
    print(f'\nSmallest N with no statistically significant difference vs. the full model (adjusted): {n_sugerido_adj}')
else:
    print('\nNenhum candidato testado ficou estatisticamente equivalente ao modelo completo (ajustada) — '
          'considere testar valores maiores.')

tab_candidatos_adj_path = f'{OUTDIR_PARTE2}/fase6_candidatos_n_delong_ajustada.csv'
tab_candidatos_n_adj.to_csv(tab_candidatos_adj_path, index=False, encoding='utf-8-sig')
print(f'Tabela salva: {tab_candidatos_adj_path}')

### Note — caveat on the N decision under the adjusted definition

Unlike the original definition (where N=28 was consistently non-significant, with p rising smoothly up to
N=30), the adjusted definition shows no clean pattern: p-values oscillate above and below 0.05 with no
clear trend (N=24: 0.007 → N=35: 0.046 → **N=40: 0.250** → N=50: 0.024 → N=60: 0.004), and the AUROC
difference stays small throughout (between −0.014 and +0.005).

**Why this happens.** The gradient boosting engine stochastically samples columns
(`colsample_bytree`/`feature_fraction`) at each tree split. Changing the total number of available
variables changes the pool from which this sampling draws, so each tested N uses a slightly different
column subset by chance — not just "more or fewer variables of the same ranking". With `random_state`
fixed but different input columns per candidate, some noise from one N to the next is expected,
especially when the real AUROC difference is already small.

**Why N=40 should not, alone, be taken as the final decision.** Testing 9 candidates at α=0.05, it is
statistically expected that at least one crosses the significance line by chance, even if the true effect
is stable — a single isolated `nao_significativo=True`, surrounded by `False` before and after, is more
consistent with sampling noise than with a real point of equivalence.

### Note — joint reading: ablation curve + N decision (adjusted definition)

The ablation curve shows the "elbow" already at n≈15–20 (AUROC jumping from 0.866 at n=5 to ~0.89), and
from n=30 (AUROC=0.901) the series oscillates within a narrow band around the full model
(n=132, AUROC=0.907): 0.901 → 0.911 (n=50) → 0.906 (n=75) → 0.911 (n=100) → 0.908 (full). There is no
monotonically rising trend with more variables — the pattern is consistent with the boosting engine's
column-sampling noise, not with real information gain.

The DeLong test singled out N=40 as the only non-significant candidate (p=0.25), surrounded by
significant ones before and after — a typical sign of sampling noise, not of a real point of equivalence.
The ablation curve agrees: there is no plateau specific to n=40 that distinguishes it from n=30, 50, 75,
or 100.

**Decision adopted.** **N=30** is kept as the simplified score's size under the adjusted definition as
well — not because DeLong cleanly confirmed it at that specific N (it did not), but because:
1. the ablation curve already shows visual stabilization from that point on;
2. keeping the same N as the original definition preserves direct comparability between the two Phase 5
   analyses (Section 10 · SHAP importance comparison), which depend on both scores having the same size;
3. adopting N=40 solely for its point significance in DeLong would trade a stable decision for one
   sensitive to which specific candidate happened to be tested.

**Recorded limitation.** Unlike the original definition, where the statistical criterion (DeLong) and the
ablation curve converged directly on N≈28–30, under the adjusted definition the N decision rests mostly
on visual reading of the ablation curve — the statistical criterion, alone, did not produce a stable cut
point in this definition, probably due to the smaller sample (transferred excluded) combined with the
inherent variance of the boosting engine's column sampling.

### Note — SHAP importance comparison between definitions (N=30)

The next panel compares the simplified score's (N=30) top-15 SHAP importance under each outcome
definition: the original model (`modelo_reduzido`, `variaveis_top`, computed in Part I) and the adjusted
model (`modelo_reduzido_adj`, `variaveis_top_adj`, previous cell). The goal is to check whether
redefining the outcome (transferred excluded) changes which variables dominate the simplified score, or
merely shifts the magnitude of mortality in the categories already identified in Sections 5 and 9.

In [ ]:
# --- SHAP importance comparison (top 15) between the original and adjusted definitions, both from the N=30 score ---
N_TOP_SHAP = 15

# Original: modelo_reduzido and variaveis_top come from Part I (cell 'Prepare the full model and the
# simplified score'). The corresponding Xtr comes from dados_prep (Part I) — not the global Xtr, which was
# overwritten by Part II Section 6's loop.
Xtr_original_top30 = dados_prep[NOME_CONJUNTO_ABL]['Xtr'][variaveis_top]
explainer_original_top30 = shap.TreeExplainer(modelo_reduzido)
shap_vals_original_top30 = explainer_original_top30(Xtr_original_top30)
importancia_original_top30 = pd.Series(
    np.abs(shap_vals_original_top30.values).mean(axis=0), index=Xtr_original_top30.columns
).sort_values(ascending=False)

# Adjusted: modelo_reduzido_adj and variaveis_top_adj, computed in the ROC-curve cell (Section 10)
explainer_ajustada_top30 = shap.TreeExplainer(modelo_reduzido_adj)
shap_vals_ajustada_top30 = explainer_ajustada_top30(Xtr_reduzido_adj)
importancia_ajustada_top30 = pd.Series(
    np.abs(shap_vals_ajustada_top30.values).mean(axis=0), index=Xtr_reduzido_adj.columns
).sort_values(ascending=False)

top15_original = importancia_original_top30.head(N_TOP_SHAP).iloc[::-1]
top15_ajustada = importancia_ajustada_top30.head(N_TOP_SHAP).iloc[::-1]

# --- Variable sets exclusive to each definition, computed BEFORE the plot (to color the bars) ---
variaveis_apenas_original = set(top15_original.index) - set(top15_ajustada.index)
variaveis_apenas_ajustada = set(top15_ajustada.index) - set(top15_original.index)

COR_COMPARTILHADA = '#1F3864'  # same color in both panels, for variables in both definitions' top 15
COR_EXCLUSIVA = '#F2A900'      # amber — only for variables exclusive to each panel

cores_original = [COR_EXCLUSIVA if v in variaveis_apenas_original else COR_COMPARTILHADA
                  for v in top15_original.index]
cores_ajustada = [COR_EXCLUSIVA if v in variaveis_apenas_ajustada else COR_COMPARTILHADA
                  for v in top15_ajustada.index]

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))

axes[0].barh(top15_original.index, top15_original.values, color=cores_original)
axes[0].set_title('Original — top 15 (N=30 score)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Mean |SHAP|')

axes[1].barh(top15_ajustada.index, top15_ajustada.values, color=cores_ajustada)
axes[1].set_title('Adjusted — top 15 (N=30 score)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Mean |SHAP|')

# --- Single legend, shared by both panels ---
from matplotlib.patches import Patch
legenda_cores = [
    Patch(facecolor=COR_COMPARTILHADA, label='Shared between the two definitions'),
    Patch(facecolor=COR_EXCLUSIVA, label="Exclusive to this definition (outside the other's top 15)"),
]
fig.legend(handles=legenda_cores, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.02),
           fontsize=9, frameon=False)

plt.suptitle('SHAP importance — top 15 variables of the simplified score (N=30)\noriginal vs. adjusted',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.04, 1, 1])  # reserves space at the bottom for the legend

fig_shap_comparacao_path = f'{OUTDIR_PARTE2}/fase6_shap_top15_original_vs_ajustada.png'
fig.savefig(fig_shap_comparacao_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura salva: {fig_shap_comparacao_path}')

print(f'\nVariables in the top 15 only under the original definition: {sorted(variaveis_apenas_original)}')
print(f'Variables in the top 15 only under the adjusted definition: {sorted(variaveis_apenas_ajustada)}')

## 11 · Synthesis and next steps (Phase 6)

This notebook consolidated, for Model A: subgroup fairness, classification metrics at three thresholds,
post-hoc recalibration, local transparency (SHAP), the TRIPOD-AI/PROBAST checklist, robustness with
repeated admissions, ablation, and the simplified score (N=30) — all under both outcome definitions,
original and adjusted, with formal quantification of transfer bias by DeLong.

**Next steps — Phase 6:**
- Close the TRIPOD-AI/PROBAST checklist.
- Formalize the risk calculator (a restricted gradient boosting model, not a point score) for Set 2, with
  a unified N, incorporating the N decision also under the adjusted definition.
- Record transfer bias as a cross-cutting methodological limitation, referenced in all Model A modeling
  phases.